In [1]:
import matplotlib as mpl
from matplotlib import font_manager
import scanpy as sc
import pandas as pd
import numpy as np
import scrublet as scr
import matplotlib.pyplot as plt
from adjustText import adjust_text

# 统一Arial字体路径（复用你最开始已经注册的arial.ttf，不再两套字体文件混乱）
arial_font_path = "/home/lishaogang/.fonts/arial.ttf"
# 注册字体
font_manager.fontManager.addfont(arial_font_path)
# 获取字体真实family名称，避免手写名称匹配失败
arial_family = font_manager.FontProperties(fname=arial_font_path).get_name()
print(f"✅ 识别字体家族: {arial_family}")

# 一次性全局配置matplotlib，后续绘图全部自动继承
mpl.rcParams.update({
    "font.family": arial_family,
    "font.sans-serif": [arial_family],
    "font.weight": "regular",
    # SCI矢量图字体嵌入（文字可编辑，防返修）
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    # 数学符号统一Arial
    "mathtext.fontset": "custom",
    "mathtext.rm": arial_family,
    "mathtext.bf": arial_family,
    "mathtext.it": arial_family,
    "mathtext.cal": arial_family,
    "mathtext.sf": arial_family,
    "mathtext.tt": arial_family,
    # 全局默认字号预设
    "axes.titlesize": 16,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9
})


In [2]:
import os
os.chdir("/home/lishaogang/NEU2026_May")

In [3]:
adata = sc.read_h5ad("GEO_data/adata_final_0610.h5ad")

In [6]:
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# =========================
# 0. 参数
# =========================
SAMPLE_COL = "orig.ident"
DISEASE_COL = "Disease"
DATASET_COL = "dataset"

COUNT_LAYER = "counts"
LOG_LAYER = "logcounts"

MIN_CELLS_PER_SAMPLE = 20
MIN_SAMPLES_PER_DISEASE = 3
N_TOP_GENES = 3000

# =========================
# 1. 从 counts 重新计算 logcounts
# =========================
if COUNT_LAYER not in adata.layers:
    raise ValueError(f"{COUNT_LAYER} not found in adata.layers")

adata_tmp = adata.copy()

# 使用 counts 作为 X
adata_tmp.X = adata_tmp.layers[COUNT_LAYER].copy()

# normalize total to 10,000
sc.pp.normalize_total(adata_tmp, target_sum=1e4)

# log1p
sc.pp.log1p(adata_tmp)

# 存入原始 adata
adata.layers[LOG_LAYER] = adata_tmp.X.copy()

print(f"Added layer: adata.layers['{LOG_LAYER}']")
print(adata.layers[LOG_LAYER].shape)

In [14]:
# 方式1：del 删除
del adata.layers['cp10k_log1p']

In [8]:
# 重命名分类，精准替换NK cell lineage
adata.obs["cell_type"] = adata.obs["cell_type"].cat.rename_categories({
    "NK cell lineage": "Innate lymphoid cell lineage"
})

# 获取原分类名
old_cats = adata.obs["cell_type"].cat.categories
# 批量去掉 " lineage"
new_cats = [cat.replace(" lineage", "") for cat in old_cats]
# 重命名分类
adata.obs["cell_type"] = adata.obs["cell_type"].cat.rename_categories(dict(zip(old_cats, new_cats)))

# 查看结果
print(adata.obs["cell_type"].value_counts())

In [10]:
# 把 obs 列里所有 "maMac" 批量替换成 "mdMac"
adata.obs['cell_subtype_short'] = adata.obs['cell_subtype_short'].cat.rename_categories({"maMac": "mdMac"})


In [11]:
# 查看所有分类名，确认 maMac 已变成 mdMac
print(adata.obs['cell_subtype_short'].cat.categories.tolist())
# 统计各类细胞数量
print(adata.obs['cell_subtype_short'].value_counts())

In [12]:
adata.obs['cell_subtype_short'] = adata.obs['cell_subtype_short'].cat.remove_unused_categories()


In [13]:
# 查看所有分类名，确认 maMac 已变成 mdMac
print(adata.obs['cell_subtype_short'].cat.categories.tolist())
# 统计各类细胞数量
print(adata.obs['cell_subtype_short'].value_counts())

In [15]:
adata

In [16]:
adata.write_h5ad("GEO_data/adata_final_0610.h5ad")

In [19]:
csf_mask = adata.obs["tissue"] == 'CSF'
adata_csf = adata[csf_mask].copy()

In [20]:
pbmc_mask = adata.obs["tissue"] != 'CSF'
adata_pbmc = adata[csf_mask].copy()

In [ ]:
adata_csf.write_h5ad("GEO_data/adata_final_0610_csf.h5ad")

In [21]:
adata_pbmc.write_h5ad("GEO_data/adata_final_0610_pbmc.h5ad")

In [13]:
adata_csf = sc.read_h5ad("GEO_data/adata_final_0610_csf.h5ad")

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib.colorbar as cbar_module

# ... 前面的 cell_type_order, marker_dict, 预处理部分保持不变 ...

# 4. 创建 DotPlot 对象
dp = sc.pl.dotplot(
    adata,
    var_names=clean_marker_dict,
    groupby='cell_subtype_short',
    standard_scale='var',
    smallest_dot=2,
    cmap='Reds',
    var_group_rotation=0,
    figsize=(15, 7),
    return_fig=True
)

# 5. 生成图
dp.make_figure()

# 6. 获取轴
axes_dict = dp.get_axes()
main_ax = axes_dict["mainplot_ax"]
arial_name = 'Arial'

# ===================== X/Y 轴标签调整 =====================
main_ax.tick_params(axis='x', labelsize=8, labelrotation=65)
main_ax.tick_params(axis='y', labelsize=9)

# ===================== ✅ 修复：颜色图例垂直 (无误伤版) =====================
if "color_legend_ax" in axes_dict:
    cl_ax = axes_dict["color_legend_ax"]
    
    # 获取主图上的 mappable (DotPlot 的 collection)
    mappable = main_ax.collections[0]
    
    # 💡 关键：❌ 删除 `cl_ax.clear()`
    # 直接在 cl_ax 轴上重新绘制一个新的垂直 Colorbar，
    # 它会自动覆盖默认的水平 Colorbar
    new_cbar = plt.colorbar(mappable, cax=cl_ax, orientation='vertical')
    
    # 设置垂直 Colorbar 的标签和字体
    new_cbar.set_label('Mean expression', fontfamily=arial_name, fontsize=9)
    new_cbar.ax.tick_params(labelsize=8)
    
    # 确保刻度线在右侧
    new_cbar.ax.yaxis.set_ticks_position('right')
    
    # 统一刻度字体
    for label in new_cbar.ax.get_yticklabels():
        label.set_fontfamily(arial_name)

# ===================== ✅ 修复：大小图例调整 =====================
if "size_legend_ax" in axes_dict:
    sz_ax = axes_dict["size_legend_ax"]
    
    # 微调位置，防止与垂直颜色条重叠
    pos = sz_ax.get_position()
    sz_ax.set_position([pos.x0 + 0.05, pos.y0, pos.width, pos.height])
    
    # 统一文本字体和大小
    for t in sz_ax.texts:
        t.set_fontfamily(arial_name)
        t.set_fontsize(8)

# ===================== ✅ 修复：顶部分组标题字体统一 (兜底) =====================
# 如果顶部的 `group_extra_ax` 还活着，在这里统一字体
if "group_extra_ax" in axes_dict:
    ge_ax = axes_dict["group_extra_ax"]
    for text in ge_ax.texts:
        text.set_fontfamily(arial_name)
        text.set_fontsize(9)
        # 💡 额外提示：这里的大括号是 matplotlib 的路径对象，通常不需要手动设置字体。

# 遍历所有轴统一标签字体（最稳妥的兜底）
for ax in axes_dict.values():
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontfamily(arial_name)

# ===================== 保存 =====================
dp.fig.savefig("dotplot_with_brackets.pdf", bbox_inches='tight')
plt.show()

In [42]:
clean_marker_dict = {
    "B/Plasma": [
        "CD19", "MS4A1", "TCL1A", "CD27", "CD38", "JCHAIN"
    ],
    "T_Naive": [
        "CD3E", "CD4","CCR7", "TCF7", "IL2RA"
    ],
    "Cytotoxic": [
        "CD8A", "GZMK", "GZMB", "PRF1", "GNLY", "NKG7", "SLC4A10"
    ],
    "MAIT/γδT": [
        "KLRB1", "TRDC","TRGC1", "IL7R"
    ],
    "NK/ILC": [
       "KIT", "GATA3", "XCL1", "FCGR3A", "CX3CR1"
    ],
    "Monocytes": [
       "CD14", "CCR2","FCGR3B", "FCN1", "VCAN"
    ],
    "Macrophages": [
        "S100A12", "C1QA", "C1QC", "APOE", "CD163", "F13A1", "LYZ"
    ],
    "InfMac": [
         "CTSS", "CD44", "GBP1","ISG15", "IFITM1"
    ],
    "mdMac": [ "FCN1", "VCAN","S100A8", "S100A9", "CD36"
   
    ],
    # "Basophil": [
    #       "CLC", "MS4A2", "CPA3", "HDC"
    # ],
    "DCs": [
       "CD1C", "CLEC10A", "FCER1A", "HLA-DRA", "AXL", "SIGLEC6"
    ]
}




In [43]:
cell_type_order = ["Naive B",  "Memory B", "Plasma cell", "CD4 Naive", "CD8 Naive", "CD4 Memory", "CD8 Memory", "Treg", "CD4 CTL", "CD8 CTL", "MAIT", "γδT", "NKT", "NK bright", "Transit. NK", "NK dim", "ILC2",
                   #"Basophil", "Transitional B",
                   "cMonocytes", "ncMonocytes", "mdMac", "trMac", "infMac", "cDC2", "pDC", "ASDC"]

In [95]:
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize





# =====================
# 1. 过滤剔除 Basophil和Transitional B
# =====================
adata_plot = adata[~adata.obs["cell_subtype_short"].isin(["Basophil", "Transitional B"]), :].copy()

# =====================
# 2. 自定义 Y 轴排序
# =====================
cell_type_order = [
    "Naive B", "Transitional B", "Memory B", "Plasma cell",
    "CD4 Naive", "CD8 Naive", "CD4 Memory", "CD8 Memory",
    "Treg", "CD4 CTL", "CD8 CTL", "MAIT", "γδT", "NKT",
    "NK bright", "Transit. NK", "NK dim", "ILC2",
    "cMonocytes", "ncMonocytes", "mdMac", "trMac", "infMac",
    "cDC2", "pDC", "ASDC"
]

# 只保留 adata_plot 中实际存在的细胞类型，避免某些类型缺失时报错
existing_cell_types = adata_plot.obs["cell_subtype_short"].unique().tolist()
cell_type_order_use = [ct for ct in cell_type_order if ct in existing_cell_types]

# =====================
# 3. 字体设置
# =====================
arial_name = "Arial"



In [96]:
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

# =====================
# 参数区：统一控制图例位置
# =====================
arial_name = "Arial"

LEGEND_Y_SHIFT = 0.18      # 两个图例整体上移幅度
CBAR_X_SHIFT = 0.00        # 颜色图例左右移动
SIZE_X_SHIFT = 0.03        # 大小图例左右移动

CBAR_WIDTH = 0.012
CBAR_HEIGHT = 0.32

SIZE_WIDTH = 0.10
SIZE_HEIGHT = 0.35


def move_resize_ax(ax, width, height, x_shift=0, y_shift=0):
    """按原轴中心移动并重设宽高"""
    pos = ax.get_position()
    cx = pos.x0 + pos.width / 2 + x_shift
    cy = pos.y0 + pos.height / 2 + y_shift
    ax.set_position([
        cx - width / 2,
        cy - height / 2,
        width,
        height
    ])


def set_font(ax, fontsize=8):
    """统一字体"""
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontfamily(arial_name)
        label.set_fontsize(fontsize)
    for text in ax.texts:
        text.set_fontfamily(arial_name)


# =====================
# 4. 创建 DotPlot 对象
# =====================
dp = sc.pl.dotplot(
    adata_plot,
    var_names=clean_marker_dict,
    groupby="cell_subtype_short",
    categories_order=cell_type_order_use,
    layer="logcounts",
    standard_scale="var",
    smallest_dot=2,
    cmap="Reds",
    colorbar_title="Scaled mean expression",
    size_title="Fraction of cells\nin group (%)",
    var_group_rotation=0,
    figsize=(15, 7),
    return_fig=True,
    show=False
)

dp.make_figure()

axes_dict = dp.get_axes()
main_ax = axes_dict["mainplot_ax"]

# =====================
# 主图字体和标签
# =====================
main_ax.tick_params(axis="x", labelsize=8, labelrotation=65)
main_ax.tick_params(axis="y", labelsize=9)

for label in main_ax.get_xticklabels():
    label.set_fontfamily(arial_name)
    label.set_horizontalalignment("right")

for label in main_ax.get_yticklabels():
    label.set_fontfamily(arial_name)

# =====================
# 颜色图例：高且窄，并整体上移
# =====================
if "color_legend_ax" in axes_dict:
    cl_ax = axes_dict["color_legend_ax"]
    cl_ax.clear()

    move_resize_ax(
        cl_ax,
        width=CBAR_WIDTH,
        height=CBAR_HEIGHT,
        x_shift=CBAR_X_SHIFT,
        y_shift=LEGEND_Y_SHIFT
    )

    sm = ScalarMappable(
        norm=Normalize(vmin=0, vmax=1),
        cmap=plt.get_cmap("Reds")
    )
    sm.set_array([])

    cbar = plt.colorbar(sm, cax=cl_ax, orientation="vertical")
    cbar.set_label(
        "Scaled mean expression",
        fontfamily=arial_name,
        fontsize=9,
        labelpad=6
    )
    cbar.ax.tick_params(labelsize=8)
    cbar.ax.yaxis.set_ticks_position("right")
    cbar.ax.yaxis.set_label_position("right")

    set_font(cbar.ax, fontsize=8)

# =====================
# 大小图例：竖排、紧凑一些，并整体上移
# =====================
if "size_legend_ax" in axes_dict:
    sz_ax = axes_dict["size_legend_ax"]
    sz_ax.clear()

    move_resize_ax(
        sz_ax,
        width=SIZE_WIDTH,
        height=SIZE_HEIGHT,
        x_shift=SIZE_X_SHIFT,
        y_shift=LEGEND_Y_SHIFT
    )

    legend_values = [1.00, 0.80, 0.60, 0.40, 0.20]
    legend_labels = ["100", "80", "60", "40", "20"]

    largest_dot = getattr(dp, "largest_dot", 200)
    smallest_dot = getattr(dp, "smallest_dot", 2)

    legend_sizes = [
        smallest_dot + v * (largest_dot - smallest_dot)
        for v in legend_values
    ]

    # 手动设置更紧凑的纵向位置
    y_positions = np.array([3.2, 2.7, 2.2, 1.7, 1.2])

    sz_ax.scatter(
        np.zeros(len(legend_values)),
        y_positions,
        s=legend_sizes,
        color="lightgray",
        edgecolor="black",
        linewidth=0.6
    )

    for y, label in zip(y_positions, legend_labels):
        sz_ax.text(
            0.35,
            y,
            label,
            va="center",
            ha="left",
            fontsize=8,
            fontfamily=arial_name
        )

    sz_ax.text(
        0,
        3.75,
        "Fraction of cells\nin group (%)",
        ha="center",
        va="bottom",
        fontsize=8,
        fontfamily=arial_name
    )

    sz_ax.set_xlim(-0.6, 1.5)
    sz_ax.set_ylim(-0.1, 4.2)
    sz_ax.axis("off")

# =====================
# 顶部分组字体
# =====================
if "group_extra_ax" in axes_dict:
    for text in axes_dict["group_extra_ax"].texts:
        text.set_fontfamily(arial_name)
        text.set_fontsize(9)

# =====================
# 保存
# =====================
dp.fig.savefig("dotplot_with_brackets.pdf", bbox_inches="tight")
dp.fig.savefig("dotplot_with_brackets.png", dpi=300, bbox_inches="tight")

plt.show()

In [139]:
# =====================
# Nature / Cell-style muted categorical color map
# =====================

soft_color_map = {

    # B lineage: muted rose / salmon / plum
    "Naive B": "#C96F6F",
    "Transitional B": "#D89A84",
    "Memory B": "#B85C68",
    "Plasma cell": "#B77AB5",

    # CD4 T cells: soft blue family
    "CD4 Naive": "#7DA7CF",
    "CD4 Memory": "#AEC9E4",
    "CD4 CTL": "#3E7C9D",

    # CD8 T / MAIT: teal-green / sage family
    "CD8 Naive": "#4F9A78",
    "MAIT": "#75B38A",
    "CD8 Memory": "#98C970",
    "CD8 CTL": "#C3D894",

    # unconventional T cells: olive / yellow-green
    "γδT": "#A6B85A",
    "NKT": "#BFB552",

    # Treg: deep muted blue
    "Treg": "#3F6BA7",

    # NK / ILC: lavender / ochre / muted yellow
    "Transit. NK": "#A995C1",
    "NK dim": "#C99235",
    "NK bright": "#D7BE66",
    "ILC2": "#DDD36A",

    # Monocytes / macrophages: dusty rose / mauve / purple
    "cMonocytes": "#C97891",
    "ncMonocytes": "#9E5F73",

    "trMac": "#D6A196",
    "infMac": "#C17DB4",
    "mdMac": "#9568B1",

    # DC / granulocytes: purple-blue / green / khaki
    "cDC2": "#B46AB0",
    "pDC": "#7F8CC1",
    "ASDC": "#B79BD0",
    "Basophil": "#91BD9E",
    "Neutrophils": "#B7AC5A"
}

In [140]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from adjustText import adjust_text

# =====================
# 0. 全局参数
# =====================
celltype_col = "cell_subtype_short"
basis = "X_umap"
arial_family = "Arial"

mpl.rcParams["font.family"] = arial_family
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

# =====================
# 1. 细胞类型顺序
# =====================
cell_type_order = [
    "Naive B", "Transitional B", "Memory B", "Plasma cell",
    "CD4 Naive", "CD8 Naive", "CD4 Memory", "CD8 Memory",
    "Treg", "CD4 CTL", "CD8 CTL", "MAIT", "γδT", "NKT",
    "NK bright", "Transit. NK", "NK dim", "ILC2",
    "cMonocytes", "ncMonocytes", "mdMac", "trMac", "infMac",
    "cDC2", "pDC", "ASDC"
]


# =====================
# 3. 设置 category 顺序并应用颜色
# =====================
adata_plot.obs[celltype_col] = adata_plot.obs[celltype_col].astype(str)

present_order = [
    ct for ct in cell_type_order
    if ct in adata_plot.obs[celltype_col].unique()
]

extra_types = [
    ct for ct in sorted(adata_plot.obs[celltype_col].unique())
    if ct not in present_order
]

final_order = present_order + extra_types

adata_plot.obs[celltype_col] = pd.Categorical(
    adata_plot.obs[celltype_col],
    categories=final_order,
    ordered=True
)

missing_colors = [ct for ct in final_order if ct not in soft_color_map]
if missing_colors:
    print("Warning: missing colors for:", missing_colors)

adata_plot.uns[f"{celltype_col}_colors"] = [
    soft_color_map.get(ct, "#BDBDBD")
    for ct in final_order
]

# =====================
# 4. 计算标签位置
# 使用 medoid，避免标签落在空白区域
# =====================
def get_group_label_positions(adata, groupby, basis="X_umap"):
    coords = pd.DataFrame(
        adata.obsm[basis],
        columns=["UMAP1", "UMAP2"],
        index=adata.obs_names
    )
    coords[groupby] = adata.obs[groupby].astype(str).values

    label_pos = {}

    for group, sub in coords.groupby(groupby, observed=True):
        xy = sub[["UMAP1", "UMAP2"]].values
        median_xy = np.median(xy, axis=0)

        dist = np.sum((xy - median_xy) ** 2, axis=1)
        label_pos[group] = xy[np.argmin(dist)]

    return label_pos


label_pos = get_group_label_positions(
    adata_plot,
    groupby=celltype_col,
    basis=basis
)

# =====================
# 5. 手动微调拥挤标签
# 数值单位为 UMAP 坐标单位
# =====================
label_nudge = {
    # NK / ILC
    "NK bright": (0.15, 0.20),
    "NK dim": (-0.15, -0.12),
    "Transit. NK": (-0.25, 0.00),
    "ILC2": (0.18, -0.05),

    # Myeloid
    "cMonocytes": (0.25, 0.00),
    "ncMonocytes": (0.10, 0.12),
    "trMac": (0.10, 0.05),
    "infMac": (0.12, -0.05),
    "mdMac": (0.00, -0.12),

    # DC
    "ASDC": (-0.10, -0.05),
    "pDC": (0.00, -0.10),
    "cDC2": (0.08, 0.00),

    # B lineage
    "Naive B": (-0.10, -0.10),
    "Transitional B": (-0.10, 0.08),
    "Memory B": (0.12, 0.00),
    "Plasma cell": (0.10, 0.08),

    # T lineage
    "CD4 Naive": (-0.15, -0.03),
    "CD4 Memory": (-0.10, 0.00),
    "CD8 Naive": (0.08, -0.08),
    "CD8 Memory": (0.08, 0.06),
    "MAIT": (0.05, -0.08),
    "γδT": (0.08, 0.05),
    "NKT": (0.08, 0.05)
}

# =====================
# 6. 绘制 UMAP
# =====================
fig, ax = plt.subplots(figsize=(8, 8))

sc.pl.umap(
    adata_plot,
    color=celltype_col,
    size=3,
    alpha=0.75,
    ax=ax,
    show=False,
    frameon=False,
    legend_loc=None,
    title=None
)

# 点 rasterized，减小 PDF/SVG 文件体积，文字仍然可编辑
for coll in ax.collections:
    coll.set_rasterized(True)

# =====================
# 7. 添加标签
# =====================
texts = []

for group in final_order:
    if group not in label_pos:
        continue

    x, y = label_pos[group]
    dx, dy = label_nudge.get(group, (0, 0))

    txt = ax.text(
        x + dx,
        y + dy,
        group,
        fontsize=10,
        fontfamily=arial_family,
        fontweight="normal",
        ha="center",
        va="center",
        color="black",
        path_effects=[
            pe.withStroke(linewidth=3.2, foreground="white")
        ]
    )
    texts.append(txt)

adjust_text(
    texts,
    ax=ax,
    expand_text=(1.12, 1.22),
    force_text=(0.25, 0.45),
    arrowprops=dict(
        arrowstyle="-",
        color="#444444",
        lw=0.75,
        alpha=0.45
    )
)

# =====================
# 8. 标题
# =====================
ax.set_title(
    "UMAP by cell type",
    fontsize=16,
    pad=18,
    fontweight="normal",
    fontfamily=arial_family
)

# =====================
# 9. 左下角 UMAP 坐标轴箭头
# 使用 axes fraction 坐标，更稳定
# =====================
arrow_x = 0.06
arrow_y = 0.06
arrow_len = 0.15

ax.annotate(
    "",
    xy=(arrow_x + arrow_len, arrow_y),
    xytext=(arrow_x, arrow_y),
    xycoords="axes fraction",
    textcoords="axes fraction",
    arrowprops=dict(
        arrowstyle="->",
        lw=2,
        color="black"
    )
)

ax.annotate(
    "",
    xy=(arrow_x, arrow_y + arrow_len),
    xytext=(arrow_x, arrow_y),
    xycoords="axes fraction",
    textcoords="axes fraction",
    arrowprops=dict(
        arrowstyle="->",
        lw=2,
        color="black"
    )
)

ax.text(
    arrow_x + arrow_len / 2,
    arrow_y - 0.025,
    "UMAP1",
    transform=ax.transAxes,
    ha="center",
    va="top",
    fontsize=10,
    fontfamily=arial_family
)

ax.text(
    arrow_x - 0.025,
    arrow_y + arrow_len / 2,
    "UMAP2",
    transform=ax.transAxes,
    ha="right",
    va="center",
    rotation=90,
    fontsize=10,
    fontfamily=arial_family
)

# =====================
# 10. 保存
# =====================
fig.savefig(
    "UMAP_cell_type_Arial_editable.pdf",
    bbox_inches="tight",
    transparent=True
)

plt.show()

In [143]:
mapping_dict = {
    'CSF': 'CSF',
    'PB': 'PBMC',
    'PBMC': 'PBMC',
    'BLD': 'PBMC'
}
adata_plot.obs['tissue'] = adata_plot.obs['tissue'].map(mapping_dict)

# 只有是分类类型才执行清理
if isinstance(adata_plot.obs['tissue'].dtype, pd.CategoricalDtype):
    adata_plot.obs['tissue'] = adata_plot.obs['tissue'].cat.remove_unused_categories()

print(adata_plot.obs['tissue'].unique())

In [144]:
adata_plot.obs['tissue'].unique()

In [146]:
# =====================
# 6. 单个 tissue 画到指定 ax
# =====================
def plot_umap_one_tissue_on_ax(
    adata,
    tissue_name,
    ax,
    tissue_col="tissue",
    celltype_col="cell_subtype_short",
    cell_type_order=None,
    soft_color_map=None,
    basis="X_umap",
    arial_family="Arial",
    global_xlim=None,
    global_ylim=None,
    title=None
):
    adata_sub = adata[adata.obs[tissue_col].astype(str) == tissue_name].copy()

    if adata_sub.n_obs == 0:
        print(f"Warning: no cells found for tissue = {tissue_name}")
        ax.set_visible(False)
        return ax

    # 当前 tissue 中存在的细胞类型
    present_order = [
        ct for ct in cell_type_order
        if ct in adata_sub.obs[celltype_col].astype(str).unique()
    ]

    extra_types = [
        ct for ct in sorted(adata_sub.obs[celltype_col].astype(str).unique())
        if ct not in present_order
    ]

    final_order = present_order + extra_types

    adata_sub.obs[celltype_col] = pd.Categorical(
        adata_sub.obs[celltype_col].astype(str),
        categories=final_order,
        ordered=True
    )

    adata_sub.uns[f"{celltype_col}_colors"] = [
        soft_color_map.get(ct, "#BDBDBD")
        for ct in final_order
    ]

    label_pos = get_group_label_positions(
        adata_sub,
        groupby=celltype_col,
        basis=basis
    )

    # 绘制 UMAP
    sc.pl.umap(
        adata_sub,
        color=celltype_col,
        size=3,
        alpha=0.75,
        ax=ax,
        show=False,
        frameon=False,
        legend_loc=None,
        title=None
    )

    # 统一 CSF / PBMC 坐标范围
    if global_xlim is not None:
        ax.set_xlim(global_xlim)
    if global_ylim is not None:
        ax.set_ylim(global_ylim)

    # 点 rasterized，文字保留可编辑
    for coll in ax.collections:
        coll.set_rasterized(True)

    # 添加标签
    texts = []

    for group in final_order:
        if group not in label_pos:
            continue

        x, y = label_pos[group]
        dx, dy = label_nudge.get(group, (0, 0))

        txt = ax.text(
            x + dx,
            y + dy,
            group,
            fontsize=9,
            fontfamily=arial_family,
            fontweight="normal",
            ha="center",
            va="center",
            color="black",
            path_effects=[
                pe.withStroke(linewidth=3.0, foreground="white")
            ]
        )
        texts.append(txt)

    adjust_text(
        texts,
        ax=ax,
        expand_text=(1.10, 1.18),
        force_text=(0.22, 0.40),
        arrowprops=dict(
            arrowstyle="-",
            color="#444444",
            lw=0.65,
            alpha=0.40
        )
    )

    # 标题
    ax.set_title(
        title if title is not None else tissue_name,
        fontsize=15,
        pad=12,
        fontweight="normal",
        fontfamily=arial_family
    )

    # 左下角 UMAP 坐标轴箭头
    arrow_x = 0.06
    arrow_y = 0.06
    arrow_len = 0.14

    ax.annotate(
        "",
        xy=(arrow_x + arrow_len, arrow_y),
        xytext=(arrow_x, arrow_y),
        xycoords="axes fraction",
        textcoords="axes fraction",
        arrowprops=dict(
            arrowstyle="->",
            lw=1.8,
            color="black"
        )
    )

    ax.annotate(
        "",
        xy=(arrow_x, arrow_y + arrow_len),
        xytext=(arrow_x, arrow_y),
        xycoords="axes fraction",
        textcoords="axes fraction",
        arrowprops=dict(
            arrowstyle="->",
            lw=1.8,
            color="black"
        )
    )

    ax.text(
        arrow_x + arrow_len / 2,
        arrow_y - 0.025,
        "UMAP1",
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=9,
        fontfamily=arial_family
    )

    ax.text(
        arrow_x - 0.025,
        arrow_y + arrow_len / 2,
        "UMAP2",
        transform=ax.transAxes,
        ha="right",
        va="center",
        rotation=90,
        fontsize=9,
        fontfamily=arial_family
    )

    return ax

In [147]:
# =====================
# 7. CSF / PBMC 并排绘制
# =====================
tissue_col = "tissue"
tissue_order = ["CSF", "PBMC"]

adata_plot.obs[celltype_col] = adata_plot.obs[celltype_col].astype(str)
adata_plot.obs[tissue_col] = adata_plot.obs[tissue_col].astype(str)

print(adata_plot.obs[tissue_col].value_counts())

# 统一 UMAP 坐标范围
global_xy = adata_plot.obsm[basis]

x_min, x_max = global_xy[:, 0].min(), global_xy[:, 0].max()
y_min, y_max = global_xy[:, 1].min(), global_xy[:, 1].max()

x_pad = (x_max - x_min) * 0.04
y_pad = (y_max - y_min) * 0.04

global_xlim = (x_min - x_pad, x_max + x_pad)
global_ylim = (y_min - y_pad, y_max + y_pad)

# 一行两列
fig, axes = plt.subplots(
    1, 2,
    figsize=(15.5, 7.5),
    sharex=True,
    sharey=True
)

for ax, tissue_name in zip(axes, tissue_order):
    plot_umap_one_tissue_on_ax(
        adata=adata_plot,
        tissue_name=tissue_name,
        ax=ax,
        tissue_col=tissue_col,
        celltype_col=celltype_col,
        cell_type_order=cell_type_order,
        soft_color_map=soft_color_map,
        basis=basis,
        arial_family=arial_family,
        global_xlim=global_xlim,
        global_ylim=global_ylim,
        title=tissue_name
    )

# 总标题可以不要，论文主图通常更干净
# fig.suptitle(
#     "UMAP by cell type",
#     fontsize=17,
#     fontfamily=arial_family,
#     y=0.98
# )

plt.subplots_adjust(
    wspace=0.04,
    left=0.02,
    right=0.98,
    top=0.92,
    bottom=0.04
)

fig.savefig(
    "UMAP_cell_type_CSF_PBMC_side_by_side_Arial_editable.pdf",
    bbox_inches="tight",
    transparent=True
)

plt.show()

In [232]:
import matplotlib.pyplot as plt
import scanpy as sc
import matplotlib as mpl

# 保证 PDF 中字体可编辑
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

fig, ax = plt.subplots(figsize=(6, 6))

sc.pl.umap(
    adata_plot,
    color="dataset",
    size=3,
    alpha=0.75,
    ax=ax,
    show=False,
    frameon=False,
    legend_loc="center left",
    title=None,
    legend_fontsize=7
)

# =====================
# 核心：只将 UMAP 点格栅化
# =====================
for coll in ax.collections:
    coll.set_rasterized(True)

# 调整图例位置
leg = ax.get_legend()
if leg is not None:
    leg.set_bbox_to_anchor((1.03, 1))
    leg.set_loc("upper left")
    leg.set_frame_on(False)

plt.tight_layout()

plt.savefig(
    "umap_dataset_rasterized.pdf",
    dpi=300,              # 控制 rasterized 点的清晰度
    bbox_inches="tight",
    transparent=True
)

plt.show()
plt.close()

In [15]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
from scipy.cluster.hierarchy import linkage, dendrogram

# --------------------------
# 函数1：计算绘图所需的所有数据（纯数据处理）
# --------------------------
def calculate_roe_heatmap_data(
    adata, 
    cell_type_col='cell_subtype_short', 
    broad_type_col='cell_type', 
    sample_col='orig.ident', 
    group_col='Disease', 
    age_col='age'
):
    """
    计算ROE热图所需的核心数据：
    包括ROE矩阵、样本/细胞亚群排序、疾病聚类结果、年龄数据等
    
    参数：
        adata: AnnData对象，包含细胞注释信息
        cell_type_col: 细胞亚群列名
        broad_type_col: 细胞大类列名
        sample_col: 样本列名
        group_col: 疾病分组列名
        age_col: 年龄列名
    
    返回：
        data_dict: 包含所有绘图数据的字典
    """
    # 1. 基础数据提取与清洗
    obs_df = adata.obs[[sample_col, group_col, cell_type_col, broad_type_col, age_col]].copy()
    # 类型转换与空值过滤
    for col in [sample_col, group_col, cell_type_col, broad_type_col]:
        obs_df[col] = obs_df[col].astype(str)
    obs_df[age_col] = pd.to_numeric(obs_df[age_col], errors='coerce')  # 年龄转数值
    obs_df = obs_df[obs_df[cell_type_col] != 'nan']  # 过滤空亚群
    
    # 2. 计算ROE (Relative Over-Expression)
    cross_tab = pd.crosstab(obs_df[cell_type_col], obs_df[sample_col])
    row_sums = cross_tab.sum(axis=1).values.reshape(-1, 1)
    col_sums = cross_tab.sum(axis=0).values.reshape(1, -1)
    total_sum = cross_tab.values.sum()
    expected = (row_sums @ col_sums) / total_sum  # 期望频数
    roe_df = np.log2((cross_tab / expected) + 0.1)  # 加0.1避免log(0)
    
    # 3. 疾病组聚类与样本排序
    # 样本-疾病/年龄映射（每个样本对应唯一疾病+年龄）
    sample_meta = obs_df[[sample_col, group_col, age_col]].drop_duplicates().set_index(sample_col)
    # 疾病组均值聚类
    group_means = roe_df.T.groupby(sample_meta[group_col]).mean()
    group_linkage = linkage(group_means, method='ward')
    group_dendro = dendrogram(group_linkage, no_plot=True)
    clustered_group_order = group_means.index[group_dendro['leaves']].tolist()  # 疾病聚类后的顺序
    # 按聚类结果排序样本
    sample_meta[group_col] = pd.Categorical(
        sample_meta[group_col], 
        categories=clustered_group_order, 
        ordered=True
    )
    sample_meta = sample_meta.sort_values(group_col)
    unique_groups = clustered_group_order  # 排序后的疾病组
    n_cols = len(sample_meta)  # 样本总数
    
    # 4. 细胞大类/亚群排序（按预设大类顺序 + 亚群聚类）
    broad_order = ['T cell lineage', 'NK cell lineage','B cell lineage', 'Myeloid cell lineage', 'pDC']
    broad_types = [b for b in broad_order if b in obs_df[broad_type_col].unique()]  # 过滤不存在的大类
    broad_to_subtypes = {}  # 大类->排序后的亚群映射
    height_ratios = []     # 每个大类对应的亚群数量（用于绘图高度比例）
    
    for bt in broad_types:
        subtypes = obs_df[obs_df[broad_type_col] == bt][cell_type_col].unique()
        sub_data = roe_df.loc[subtypes, sample_meta.index]
        # 亚群内聚类排序
        if len(sub_data) > 1:
            Z = linkage(sub_data, 'ward')
            reorder_idx = dendrogram(Z, no_plot=True)['leaves']
            sorted_sub = sub_data.index[reorder_idx[::-1]].tolist()
        else:
            sorted_sub = sub_data.index.tolist()
        broad_to_subtypes[bt] = sorted_sub
        height_ratios.append(len(sorted_sub))
    
    # 5. 提取年龄数据（绘图用）
    age_values = sample_meta[age_col].values.reshape(1, -1)  # 排序后的样本年龄
    # 计算疾病组宽度/中心位置（用于疾病Bar和聚类树）
    group_widths = [sum(sample_meta[group_col] == g) for g in unique_groups]
    group_centers = []
    x_pos = 0
    for width in group_widths:
        group_centers.append(x_pos + width/2)
        x_pos += width
    
    # 6. 整理返回数据
    data_dict = {
        # 核心矩阵
        'roe_df': roe_df,
        # 样本/分组信息
        'sample_meta': sample_meta,          # 排序后的样本元数据
        'unique_groups': unique_groups,      # 聚类后的疾病组顺序
        'group_widths': group_widths,        # 每个疾病组的样本数
        'group_centers': group_centers,      # 每个疾病组的中心位置
        'n_cols': n_cols,                    # 样本总数（列数）
        # 细胞亚群信息
        'broad_types': broad_types,          # 细胞大类列表
        'broad_to_subtypes': broad_to_subtypes,  # 大类->亚群映射
        'height_ratios': height_ratios,      # 每个大类的亚群数（高度比例）
        # 聚类结果
        'group_dendro': group_dendro,        # 疾病组聚类树数据
        'max_h': np.array(group_dendro['dcoord']).max(),  # 聚类树最大高度
        # 年龄数据
        'age_values': age_values,            # 排序后的样本年龄
        'age_min': obs_df[age_col].min(),    # 年龄最小值（色条用）
        'age_max': obs_df[age_col].max(),    # 年龄最大值（色条用）
        # 原始参数（绘图时复用）
        'cell_type_col': cell_type_col,
        'broad_type_col': broad_type_col,
        'sample_col': sample_col,
        'group_col': group_col,
        'age_col': age_col
    }
    return data_dict



In [16]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, dendrogram
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec


def plot_roe_heatmap1(
    data_dict,
    cell_size=0.35,
    vmax=10,
    roe_colors=None,
    age_cmap="YlGnBu",
    group_palette="Set2",

    disease_order=None,
    cell_order=None,

    add_unlisted_groups=True,
    add_unlisted_celltypes=True,

    # 强制 Y 轴细胞类型聚类
    row_cluster=True,
    row_cluster_method="ward",
    show_row_dendrogram=True,

    # 疾病是否显示底部聚类树
    show_group_dendrogram_if_clustered=True,

    save_path=None,
    dpi=300,
    cbar_x=0.80,
    grey_cns="#333333",

    # 不画白色分隔线
    show_group_separators=False,

    # 透明导出
    transparent=True,
    pad_inches=0.02
):
    """
    ROE heatmap.

    修改版：
    1. 全图透明背景，不使用 white facecolor
    2. 所有 axes 无边框、无背景 patch
    3. 不画白色 group 分隔线
    4. Y 轴细胞类型默认强制聚类
    5. 色条、年龄条、疾病条、热图处于同一透明画布视觉层
    """

    if roe_colors is None:
        roe_colors = [
            "#244773",
            "#73A9D0",
            "#F8F8F8",
            "#D4887A",
            "#752C36"
        ]

    # ------------------------------------------------------------------
    # 小工具：去除所有 axes 背景、边框、网格
    # ------------------------------------------------------------------
    def clean_axis(ax):
        ax.set_facecolor("none")
        ax.patch.set_alpha(0)
        ax.patch.set_visible(False)
        ax.set_frame_on(False)

        for spine in ax.spines.values():
            spine.set_visible(False)

        ax.grid(False)

        return ax

    # =====================
    # 1. Extract data
    # =====================
    roe_df = data_dict["roe_df"].copy()
    sample_meta = data_dict["sample_meta"].copy()
    age_values = data_dict["age_values"]

    group_col = data_dict["group_col"]
    age_min = data_dict["age_min"]
    age_max = data_dict["age_max"]

    original_sample_order = sample_meta.index.tolist()

    # =====================
    # 2. Disease order
    # =====================
    if disease_order is not None:
        existing_groups = (
            sample_meta[group_col]
            .astype(str)
            .unique()
            .tolist()
        )

        ordered_groups = [
            g for g in disease_order
            if g in existing_groups
        ]

        missing_groups = [
            g for g in disease_order
            if g not in existing_groups
        ]

        if missing_groups:
            print("Warning: groups in disease_order not found:", missing_groups)

        if add_unlisted_groups:
            unlisted_groups = [
                g for g in existing_groups
                if g not in ordered_groups
            ]
            unique_groups = ordered_groups + unlisted_groups
        else:
            unique_groups = ordered_groups

        sample_order = []

        for g in unique_groups:
            sample_order.extend(
                sample_meta.index[
                    sample_meta[group_col].astype(str) == g
                ].tolist()
            )

        sample_meta = sample_meta.loc[sample_order].copy()
        roe_df = roe_df.loc[:, sample_order]

        group_widths = [
            int(np.sum(sample_meta[group_col].astype(str) == g))
            for g in unique_groups
        ]

        group_centers = []
        x_pos = 0

        for width in group_widths:
            group_centers.append(x_pos + width / 2)
            x_pos += width

        show_group_dendro = False
        max_h = 1

    else:
        unique_groups = list(data_dict["unique_groups"])
        group_widths = data_dict["group_widths"]
        group_centers = data_dict["group_centers"]
        group_dendro = data_dict["group_dendro"]
        max_h = data_dict["max_h"]

        sample_order = sample_meta.index.tolist()
        roe_df = roe_df.loc[:, sample_order]

        show_group_dendro = show_group_dendrogram_if_clustered

    n_cols = len(sample_meta)

    # =====================
    # 3. Reorder age values
    # =====================
    def reorder_age_values(age_values, old_order, new_order):
        if isinstance(age_values, pd.Series):
            return age_values.loc[new_order].to_frame().T

        if isinstance(age_values, pd.DataFrame):
            if set(new_order).issubset(set(age_values.columns)):
                return age_values.loc[:, new_order]

            if set(new_order).issubset(set(age_values.index)):
                return age_values.loc[new_order].T

        arr = np.asarray(age_values)

        if arr.ndim == 1:
            arr = arr.reshape(1, -1)

        old_pos = {
            s: i for i, s in enumerate(old_order)
        }

        reorder_idx = [
            old_pos[s] for s in new_order
        ]

        if arr.shape[1] == len(old_order):
            return arr[:, reorder_idx]

        if arr.shape[0] == len(old_order):
            return arr[reorder_idx, :].T

        raise ValueError(
            "Cannot reorder age_values. Please check whether age_values is aligned with sample_meta."
        )

    age_values = reorder_age_values(
        age_values,
        old_order=original_sample_order,
        new_order=sample_meta.index.tolist()
    )

    # =====================
    # 4. Cell-type order
    #    现在默认强制 Y 轴聚类
    # =====================
    existing_celltypes = roe_df.index.unique().tolist()

    if cell_order is not None:
        ordered_celltypes = [
            ct for ct in cell_order
            if ct in existing_celltypes
        ]

        missing_celltypes = [
            ct for ct in cell_order
            if ct not in existing_celltypes
        ]

        if missing_celltypes:
            print("Warning: cell types in cell_order not found:", missing_celltypes)

        if add_unlisted_celltypes:
            unlisted_celltypes = [
                ct for ct in existing_celltypes
                if ct not in ordered_celltypes
            ]
            base_celltypes = ordered_celltypes + unlisted_celltypes
        else:
            base_celltypes = ordered_celltypes
    else:
        base_celltypes = existing_celltypes

    sub_data_raw = roe_df.loc[
        base_celltypes,
        sample_meta.index
    ]

    if row_cluster and sub_data_raw.shape[0] > 1:
        Z_row = linkage(
            sub_data_raw.values,
            method=row_cluster_method
        )

        row_leaves = dendrogram(
            Z_row,
            no_plot=True
        )["leaves"]

        # 保持聚类后的视觉顺序
        subtypes_order = (
            sub_data_raw
            .index[row_leaves[::-1]]
            .tolist()
        )

        show_cell_dendro = show_row_dendrogram

    else:
        Z_row = None
        subtypes_order = sub_data_raw.index.tolist()
        show_cell_dendro = False

    sub_data = roe_df.loc[
        subtypes_order,
        sample_meta.index
    ]

    n_sub = sub_data.shape[0]

    # =====================
    # 5. Figure layout
    # =====================
    heatmap_width = n_cols * cell_size
    heatmap_height = n_sub * cell_size

    left_side = 0.6
    tree_side = 1.0 if show_cell_dendro else 0.25
    right_side = 3.2
    bottom_side = 5.0 if show_group_dendro else 3.4
    top_side = 0.35

    fig_width = heatmap_width + left_side + tree_side + right_side
    fig_height = heatmap_height + bottom_side + top_side

    fig = plt.figure(
        figsize=(fig_width, fig_height),
        facecolor="none"
    )

    fig.patch.set_alpha(0)
    fig.patch.set_visible(False)

    bottom_tree_ratio = 2.5 if show_group_dendro else 0.5

    main_gs = GridSpec(
        4,
        4,
        height_ratios=[
            n_sub,
            0.35,
            0.55,
            bottom_tree_ratio
        ],
        width_ratios=[
            0.04,
            0.09 if show_cell_dendro else 0.01,
            0.74,
            0.13
        ],
        hspace=0.10,
        wspace=0.01
    )

    my_cmap = LinearSegmentedColormap.from_list(
        "roe_cmap",
        list(
            zip(
                [0.0, 0.35, 0.5, 0.65, 1.0],
                roe_colors
            )
        )
    )

    # =====================
    # 6. Left cell-type dendrogram
    # =====================
    ax_tree = fig.add_subplot(main_gs[0, 1])
    clean_axis(ax_tree)

    if show_cell_dendro and Z_row is not None and n_sub > 1:
        ddata_row = dendrogram(
            Z_row,
            orientation="left",
            ax=ax_tree,
            color_threshold=0,
            above_threshold_color=grey_cns,
            no_labels=True
        )

        max_d = np.max(ddata_row["dcoord"])
        ax_tree.set_xlim(max_d * 1.05, 0)
        ax_tree.set_ylim(0, 10 * n_sub)

        for coll in ax_tree.collections:
            coll.set_linewidth(0.55)
            coll.set_color(grey_cns)
            coll.set_zorder(2)

    ax_tree.axis("off")

    # =====================
    # 7. Main heatmap
    # =====================
    ax_heatmap = fig.add_subplot(main_gs[0, 2])
    clean_axis(ax_heatmap)

    sns.heatmap(
        sub_data,
        ax=ax_heatmap,
        cmap=my_cmap,
        center=0,
        vmin=-vmax,
        vmax=vmax,
        cbar=False,
        xticklabels=False,
        yticklabels=True,
        linewidths=0,
        linecolor="none",
        square=False
    )

    clean_axis(ax_heatmap)

    ax_heatmap.set_xlabel("")
    ax_heatmap.set_ylabel("")

    ax_heatmap.yaxis.tick_right()
    ax_heatmap.yaxis.set_label_position("right")
    ax_heatmap.set_ylim(n_sub, 0)
    ax_heatmap.grid(False)

    ax_heatmap.tick_params(
        axis="both",
        length=0,
        width=0
    )

    plt.setp(
        ax_heatmap.get_yticklabels(),
        rotation=0,
        horizontalalignment="left",
        fontsize=12,
        fontweight="medium"
    )

    # 不再画白色分隔线
    if show_group_separators:
        curr_pos = 0

        for group in unique_groups[:-1]:
            curr_pos += np.sum(
                sample_meta[group_col].astype(str) == group
            )

            ax_heatmap.axvline(
                curr_pos,
                color=grey_cns,
                linewidth=0.4,
                alpha=0.5,
                zorder=3
            )

    # =====================
    # 8. Age annotation bar
    # =====================
    ax_age = fig.add_subplot(main_gs[1, 2])
    clean_axis(ax_age)

    sns.heatmap(
        age_values,
        ax=ax_age,
        cmap=age_cmap,
        cbar=False,
        xticklabels=False,
        yticklabels=False,
        linewidths=0,
        linecolor="none"
    )

    clean_axis(ax_age)

    ax_age.set_xlabel("")
    ax_age.set_ylabel("")
    ax_age.tick_params(
        axis="both",
        length=0,
        width=0
    )

    if show_group_separators:
        curr_pos = 0

        for group in unique_groups[:-1]:
            curr_pos += np.sum(
                sample_meta[group_col].astype(str) == group
            )

            ax_age.axvline(
                curr_pos,
                color=grey_cns,
                linewidth=0.4,
                alpha=0.5,
                zorder=3
            )

    # =====================
    # 9. Disease annotation bar
    # =====================
    ax_bar = fig.add_subplot(main_gs[2, 2])
    clean_axis(ax_bar)

    group_colors = sns.color_palette(
        group_palette,
        len(unique_groups)
    )

    color_dict = dict(
        zip(
            unique_groups,
            group_colors
        )
    )

    x_pos = 0

    for group, width in zip(unique_groups, group_widths):
        ax_bar.barh(
            0,
            width,
            left=x_pos,
            color=color_dict[group],
            height=0.8,
            edgecolor="none",
            linewidth=0,
            zorder=2
        )

        ax_bar.text(
            x_pos + width / 2,
            -0.8,
            group,
            ha="right",
            va="top",
            rotation=45,
            fontweight="bold",
            fontsize=14,
            zorder=3
        )

        x_pos += width

    ax_bar.set_xlim(0, n_cols)
    ax_bar.set_ylim(-1.5, 1.0)
    ax_bar.axis("off")
    clean_axis(ax_bar)

    # =====================
    # 10. Bottom disease dendrogram
    # =====================
    ax_bottom_tree = fig.add_subplot(main_gs[3, 2])
    clean_axis(ax_bottom_tree)

    if show_group_dendro:
        icoord = np.array(group_dendro["icoord"])
        dcoord = np.array(group_dendro["dcoord"])

        default_leaf_pos = np.array([
            j * 10 + 5
            for j in range(len(unique_groups))
        ])

        def map_group_coords(x):
            return np.interp(
                x,
                default_leaf_pos,
                group_centers
            )

        for xs, ys in zip(icoord, dcoord):
            ax_bottom_tree.plot(
                map_group_coords(xs),
                max_h - ys,
                color=grey_cns,
                lw=0.55,
                zorder=2
            )

        ax_bottom_tree.set_xlim(0, n_cols)
        ax_bottom_tree.set_ylim(
            -max_h * 0.1,
            max_h * 1.1
        )

    ax_bottom_tree.axis("off")
    clean_axis(ax_bottom_tree)

    # =====================
    # 11. Colorbars
    # =====================
    cbar_ax = fig.add_axes([
        cbar_x,
        0.57,
        0.013,
        0.22
    ])

    clean_axis(cbar_ax)

    sm = plt.cm.ScalarMappable(
        cmap=my_cmap,
        norm=plt.Normalize(
            -vmax,
            vmax
        )
    )

    cb = plt.colorbar(
        sm,
        cax=cbar_ax,
        ticks=np.linspace(-vmax, vmax, 5)
    )

    cb.ax.set_title(
        "Log2ROE",
        fontweight="bold",
        fontsize=12,
        pad=15
    )

    cb.ax.tick_params(
        labelsize=10,
        length=0,
        width=0
    )

    cb.outline.set_visible(False)
    clean_axis(cb.ax)

    for spine in cb.ax.spines.values():
        spine.set_visible(False)

    age_cbar_ax = fig.add_axes([
        cbar_x,
        0.34,
        0.013,
        0.16
    ])

    clean_axis(age_cbar_ax)

    sm_age = plt.cm.ScalarMappable(
        cmap=age_cmap,
        norm=plt.Normalize(
            age_min,
            age_max
        )
    )

    cb_age = plt.colorbar(
        sm_age,
        cax=age_cbar_ax
    )

    cb_age.ax.set_title(
        "Age",
        fontweight="bold",
        fontsize=12,
        pad=10
    )

    cb_age.ax.tick_params(
        labelsize=10,
        length=0,
        width=0
    )

    cb_age.outline.set_visible(False)
    clean_axis(cb_age.ax)

    for spine in cb_age.ax.spines.values():
        spine.set_visible(False)

    # =====================
    # 12. Final clean
    # =====================
    for ax in fig.axes:
        clean_axis(ax)

    # =====================
    # 13. Save
    # =====================
    if save_path is not None:
        plt.savefig(
            save_path,
            dpi=dpi,
            bbox_inches="tight",
            pad_inches=pad_inches,
            transparent=transparent,
            facecolor="none",
            edgecolor="none"
        )

        print(f"Plot saved to: {save_path}")

    plt.show()

In [169]:
adata_csf = adata_plot[adata_plot.obs['tissue']=="CSF"].copy()
adata_pbmc = adata_plot[adata_plot.obs['tissue']!="CSF"].copy()

In [23]:
adata_csf = adata_csf[~adata_csf.obs['cell_subtype_short'].isin(["ncMonocytes","Basophil","Transitional B"])].copy()

In [ ]:
# 1. 先计算数据（可单独调试/保存数据）
data_dict_csf = calculate_roe_heatmap_data(
    adata=adata_csf, 
    age_col='age',  # 仅指定自定义的年龄列，其他用默认
    cell_type_col='cell_subtype_short'
)

data_dict_pbmc = calculate_roe_heatmap_data(
    adata=adata_pbmc, 
    age_col='age',  # 仅指定自定义的年龄列，其他用默认
    cell_type_col='cell_subtype_short'
)


In [26]:
plot_roe_heatmap1(
    data_dict=data_dict_csf,
    cell_size=0.35,  # 单元格大小
    vmax=10,         # ROE色条最大值
    roe_colors = ["#244773", "#73A9D0", "#F8F8F8", "#D4887A", "#752C36"],  # 自定义ROE配色
    save_path='roe_heatmap_adata_csf.pdf',  # 保存路径（支持pdf/png/jpg等格式）
    dpi=300,  # 保存分辨率（300足够期刊使用）
    cbar_x=0.82   
)

In [175]:
plot_roe_heatmap1(
    data_dict=data_dict_pbmc,
    cell_size=0.35,  # 单元格大小
    vmax=10,         # ROE色条最大值
    roe_colors = ["#244773", "#73A9D0", "#F8F8F8", "#D4887A", "#752C36"],  # 自定义ROE配色
    save_path='roe_heatmap_adata_pbmc.pdf',  # 保存路径（支持pdf/png/jpg等格式）
    dpi=300,  # 保存分辨率（300足够期刊使用）
    cbar_x=0.82   
)

In [176]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import zscore
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import re  # 用于清理文件名特殊字符

# ===================== 新增：清理文件名特殊字符的函数 =====================
def clean_filename(name):
    """
    替换文件名中的非法字符（/ \ : * ? " < > |）为下划线，避免保存报错
    """
    illegal_chars = r'[\/:*?"<>|]'
    clean_name = re.sub(illegal_chars, '_', name)
    return clean_name

# ===================== 步骤1：构建样本×细胞亚型的特征矩阵（核心） =====================
def build_sample_subtype_matrix(adata, sample_col='orig.ident', subtype_col='cell_subtype_short'):
    """
    构建样本级别的细胞亚型计数矩阵（行=样本，列=细胞亚型，值=该样本中该亚型的细胞数）
    """
    # 1. 按样本+细胞亚型统计细胞数
    sample_subtype_counts = adata.obs.groupby([sample_col, subtype_col]).size().reset_index(name='count')
    # 2. 透视成矩阵（样本为行，细胞亚型为列）
    sample_subtype_matrix = sample_subtype_counts.pivot(
        index=sample_col, columns=subtype_col, values='count'
    ).fillna(0)  # 无该亚型的样本填0
    # 3. 标准化（避免细胞总数差异影响，转为比例）
    sample_subtype_matrix = sample_subtype_matrix.div(sample_subtype_matrix.sum(axis=1), axis=0)
    return sample_subtype_matrix


# ===================== 步骤2：定义离群值识别函数（基于PCA坐标的Z-score） =====================
def detect_outliers_pca(pca_coords, z_threshold=1.5):  # 阈值调低到1.5，识别更多离群点
    """
    基于PCA前两维坐标的Z-score识别离群值
    :param pca_coords: PCA降维后的坐标矩阵（行=样本，列=PC1/PC2）
    :param z_threshold: Z-score阈值（值越低，识别的离群点越多）
    :return: 离群样本列表、正常样本列表、Z-score数组
    """
    # 计算每个样本在PC1+PC2上的综合Z-score（欧氏距离的Z-score）
    pca_dist = np.sqrt(pca_coords['PC1']**2 + pca_coords['PC2']**2)
    pca_z = zscore(pca_dist)
    # 标记离群值（阈值调低，更多样本被标记）
    outliers = pca_coords.index[pca_z > z_threshold].tolist()
    normals = pca_coords.index[pca_z <= z_threshold].tolist()
    return outliers, normals, pca_z




In [177]:
# 生成样本-细胞亚型矩阵
sample_subtype_mat = build_sample_subtype_matrix(
    adata_csf,
    sample_col='orig.ident',  # 样本ID列（根据你的数据调整）
    subtype_col='cell_subtype_short'  # 细胞亚型列
)

# ===================== 步骤3：批量处理疾病数据，准备绘图所需信息 =====================
# 3.1 提取样本-疾病映射关系
sample_disease_map = adata_csf.obs[['orig.ident', 'Disease']].drop_duplicates()
sample_disease_map = sample_disease_map.set_index('orig.ident')['Disease'].to_dict()

# 3.2 筛选有效疾病（样本数≥5）并收集绘图数据
valid_diseases = []
pca_plot_data = {}  # 存储每个疾病的PCA绘图数据
all_outliers = {}   # 存储所有疾病的离群样本

diseases = adata_csf.obs['Disease'].unique()
for disease in diseases:
    disease_samples = [s for s in sample_subtype_mat.index if sample_disease_map.get(s) == disease]
    if len(disease_samples) < 5:
        print(f"⚠️  疾病 {disease} 样本数不足5个，跳过PCA绘图")
        continue
    valid_diseases.append(disease)
    
    # 提取该疾病的样本-细胞亚型矩阵并标准化
    disease_mat = sample_subtype_mat.loc[disease_samples]
    scaler = StandardScaler()
    disease_mat_scaled = scaler.fit_transform(disease_mat)
    
    # 做PCA
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(disease_mat_scaled)
    pca_coords = pd.DataFrame(
        pca_result, index=disease_samples, columns=['PC1', 'PC2']
    )
    
    # 关键修改：z_threshold=1.5（更低的阈值=更多离群点）
    outliers, normals, pca_z = detect_outliers_pca(pca_coords, z_threshold=1.5)
    all_outliers[disease] = outliers
    print(f"🔍 疾病 {disease} 离群样本: {outliers} (共{len(outliers)}个)")
    
    # 保存绘图所需数据
    pca_plot_data[disease] = {
        'pca_coords': pca_coords,
        'normals': normals,
        'outliers': outliers,
        'explained_var': pca.explained_variance_ratio_
    }

# ===================== 步骤4：绘制合并的PCA子图并保存为PDF =====================
# 4.1 计算子图网格大小（按行列均匀分布）
n_diseases = len(valid_diseases)
n_cols = 3  # 列数，可根据需要调整
n_rows = (n_diseases + n_cols - 1) // n_cols  # 向上取整计算行数

# 4.2 创建画布（设置合适的尺寸，保证每个子图清晰）
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 5*n_rows), dpi=100)
axes = axes.flatten()  # 展平轴数组，方便遍历

# 4.3 逐个绘制子图（优化图例框+更多离群点）
for idx, disease in enumerate(valid_diseases):
    ax = axes[idx]
    plot_data = pca_plot_data[disease]
    
    # 绘制正常样本
    ax.scatter(
        plot_data['pca_coords'].loc[plot_data['normals'], 'PC1'],
        plot_data['pca_coords'].loc[plot_data['normals'], 'PC2'],
        c='skyblue', label='Normal', s=60, alpha=0.8
    )
    
    # 绘制离群样本（更多样本被标记）
    if plot_data['outliers']:
        ax.scatter(
            plot_data['pca_coords'].loc[plot_data['outliers'], 'PC1'],
            plot_data['pca_coords'].loc[plot_data['outliers'], 'PC2'],
            c='red', label='Outlier', s=80, edgecolors='black'
        )
        # 标注离群样本ID（如果太多，可注释掉避免拥挤）
        for outlier in plot_data['outliers']:
            ax.text(
                plot_data['pca_coords'].loc[outlier, 'PC1'] + 0.05,
                plot_data['pca_coords'].loc[outlier, 'PC2'] + 0.05,
                outlier, fontsize=8, color='red'
            )
    
    # 子图美化
    ev1, ev2 = plot_data['explained_var']
    ax.set_title(f'{disease}\nPC1: {ev1:.1%}, PC2: {ev2:.1%}', fontsize=10)
    ax.set_xlabel(f'PC1 ({ev1:.1%})', fontsize=8)
    ax.set_ylabel(f'PC2 ({ev2:.1%})', fontsize=8)
    
    # 关键修改：图例框更宽松（增加内边距+更大的框）
    ax.legend(
        fontsize=8,
        frameon=True,        # 显示图例边框
        framealpha=1,        # 边框不透明
        edgecolor='black',   # 边框颜色
        facecolor='white',   # 图例背景色
        borderpad=1.2,       # 内边距（调大=框更宽松）
        labelspacing=0.8,    # 标签之间的间距
        fancybox=False       # 直角边框（更清晰）
    )
    
    ax.grid(alpha=0.3)

# 4.4 隐藏多余的子图（如果有）
for idx in range(n_diseases, len(axes)):
    axes[idx].set_visible(False)

# 4.5 调整子图间距，避免重叠
plt.tight_layout()

# 4.6 保存为PDF（清理文件名，避免报错）
save_path = 'combined_disease_pca_more_outliers.pdf'
plt.savefig(
    save_path, 
    dpi=300, 
    bbox_inches='tight',
    facecolor='white',
    edgecolor='none'
)
print(f"✅ 合并的PCA图（更多离群点）已保存为：{save_path}")

# 可选：显示图片（如果需要）
plt.show()

# ===================== 步骤5：输出所有疾病的离群样本汇总 =====================
print("\n=== 所有疾病的离群样本汇总（更多离群点）===")
for disease, outliers in all_outliers.items():
    if outliers:
        print(f"{disease}: {outliers} (共{len(outliers)}个)")
    else:
        print(f"{disease}: 无离群样本")

# ===================== 新增：生成所有需剔除的离群样本统一list（去重） =====================
# 1. 汇总所有疾病的离群样本
all_outlier_samples = []
for disease_outliers in all_outliers.values():
    all_outlier_samples.extend(disease_outliers)

# 2. 去重（避免同一样本被多个疾病标记为离群，重复出现在list中）
remove_list_csf = list(set(all_outlier_samples))

# 3. 排序（可选，让list更规整）
remove_list_csf.sort()

# 4. 输出结果
print("\n=== 所有需剔除的离群样本最终list ===")
print(f"需剔除样本总数：{len(remove_list_csf)}")
print(f"remove_list_csf = {remove_list_csf}")

In [178]:
# 生成样本-细胞亚型矩阵
sample_subtype_mat = build_sample_subtype_matrix(
    adata_pbmc,
    sample_col='orig.ident',  # 样本ID列（根据你的数据调整）
    subtype_col='cell_subtype_short'  # 细胞亚型列
)

# ===================== 步骤3：批量处理疾病数据，准备绘图所需信息 =====================
# 3.1 提取样本-疾病映射关系
sample_disease_map = adata_pbmc.obs[['orig.ident', 'Disease']].drop_duplicates()
sample_disease_map = sample_disease_map.set_index('orig.ident')['Disease'].to_dict()

# 3.2 筛选有效疾病（样本数≥5）并收集绘图数据
valid_diseases = []
pca_plot_data = {}  # 存储每个疾病的PCA绘图数据
all_outliers = {}   # 存储所有疾病的离群样本

diseases = adata_csf.obs['Disease'].unique()
for disease in diseases:
    disease_samples = [s for s in sample_subtype_mat.index if sample_disease_map.get(s) == disease]
    if len(disease_samples) < 5:
        print(f"⚠️  疾病 {disease} 样本数不足5个，跳过PCA绘图")
        continue
    valid_diseases.append(disease)
    
    # 提取该疾病的样本-细胞亚型矩阵并标准化
    disease_mat = sample_subtype_mat.loc[disease_samples]
    scaler = StandardScaler()
    disease_mat_scaled = scaler.fit_transform(disease_mat)
    
    # 做PCA
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(disease_mat_scaled)
    pca_coords = pd.DataFrame(
        pca_result, index=disease_samples, columns=['PC1', 'PC2']
    )
    
    # 关键修改：z_threshold=1.5（更低的阈值=更多离群点）
    outliers, normals, pca_z = detect_outliers_pca(pca_coords, z_threshold=1.5)
    all_outliers[disease] = outliers
    print(f"🔍 疾病 {disease} 离群样本: {outliers} (共{len(outliers)}个)")
    
    # 保存绘图所需数据
    pca_plot_data[disease] = {
        'pca_coords': pca_coords,
        'normals': normals,
        'outliers': outliers,
        'explained_var': pca.explained_variance_ratio_
    }

# ===================== 步骤4：绘制合并的PCA子图并保存为PDF =====================
# 4.1 计算子图网格大小（按行列均匀分布）
n_diseases = len(valid_diseases)
n_cols = 3  # 列数，可根据需要调整
n_rows = (n_diseases + n_cols - 1) // n_cols  # 向上取整计算行数

# 4.2 创建画布（设置合适的尺寸，保证每个子图清晰）
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 5*n_rows), dpi=100)
axes = axes.flatten()  # 展平轴数组，方便遍历

# 4.3 逐个绘制子图（优化图例框+更多离群点）
for idx, disease in enumerate(valid_diseases):
    ax = axes[idx]
    plot_data = pca_plot_data[disease]
    
    # 绘制正常样本
    ax.scatter(
        plot_data['pca_coords'].loc[plot_data['normals'], 'PC1'],
        plot_data['pca_coords'].loc[plot_data['normals'], 'PC2'],
        c='skyblue', label='Normal', s=60, alpha=0.8
    )
    
    # 绘制离群样本（更多样本被标记）
    if plot_data['outliers']:
        ax.scatter(
            plot_data['pca_coords'].loc[plot_data['outliers'], 'PC1'],
            plot_data['pca_coords'].loc[plot_data['outliers'], 'PC2'],
            c='red', label='Outlier', s=80, edgecolors='black'
        )
        # 标注离群样本ID（如果太多，可注释掉避免拥挤）
        for outlier in plot_data['outliers']:
            ax.text(
                plot_data['pca_coords'].loc[outlier, 'PC1'] + 0.05,
                plot_data['pca_coords'].loc[outlier, 'PC2'] + 0.05,
                outlier, fontsize=8, color='red'
            )
    
    # 子图美化
    ev1, ev2 = plot_data['explained_var']
    ax.set_title(f'{disease}\nPC1: {ev1:.1%}, PC2: {ev2:.1%}', fontsize=10)
    ax.set_xlabel(f'PC1 ({ev1:.1%})', fontsize=8)
    ax.set_ylabel(f'PC2 ({ev2:.1%})', fontsize=8)
    
    # 关键修改：图例框更宽松（增加内边距+更大的框）
    ax.legend(
        fontsize=8,
        frameon=True,        # 显示图例边框
        framealpha=1,        # 边框不透明
        edgecolor='black',   # 边框颜色
        facecolor='white',   # 图例背景色
        borderpad=1.2,       # 内边距（调大=框更宽松）
        labelspacing=0.8,    # 标签之间的间距
        fancybox=False       # 直角边框（更清晰）
    )
    
    ax.grid(alpha=0.3)

# 4.4 隐藏多余的子图（如果有）
for idx in range(n_diseases, len(axes)):
    axes[idx].set_visible(False)

# 4.5 调整子图间距，避免重叠
plt.tight_layout()

# 4.6 保存为PDF（清理文件名，避免报错）
save_path = 'combined_disease_pca_more_outliers_pbmc.pdf'
plt.savefig(
    save_path, 
    dpi=300, 
    bbox_inches='tight',
    facecolor='white',
    edgecolor='none'
)
print(f"✅ 合并的PCA图（更多离群点）已保存为：{save_path}")

# 可选：显示图片（如果需要）
plt.show()

# ===================== 步骤5：输出所有疾病的离群样本汇总 =====================
print("\n=== 所有疾病的离群样本汇总（更多离群点）===")
for disease, outliers in all_outliers.items():
    if outliers:
        print(f"{disease}: {outliers} (共{len(outliers)}个)")
    else:
        print(f"{disease}: 无离群样本")

# ===================== 新增：生成所有需剔除的离群样本统一list（去重） =====================
# 1. 汇总所有疾病的离群样本
all_outlier_samples = []
for disease_outliers in all_outliers.values():
    all_outlier_samples.extend(disease_outliers)

# 2. 去重（避免同一样本被多个疾病标记为离群，重复出现在list中）
remove_list_pbmc = list(set(all_outlier_samples))

# 3. 排序（可选，让list更规整）
remove_list_pbmc.sort()

# 4. 输出结果
print("\n=== 所有需剔除的离群样本最终list ===")
print(f"需剔除样本总数：{len(remove_list_pbmc)}")
print(f"remove_list_pbmc = {remove_list_pbmc}")

In [179]:
remove_list = remove_list_csf +  remove_list_pbmc

In [180]:
adata_csf_clean_samples = adata_csf[~adata_csf.obs['orig.ident'].isin(remove_list_csf)].copy()
 = adata_pbmc[~adata_pbmc.obs['orig.ident'].isin(remove_list_pbmc)].copy()

In [253]:
adata_pbmc_clean_samples.obs['Disease'].unique()

In [259]:
adata_pbmc_clean_samples = adata_pbmc_clean_samples[~adata_pbmc_clean_samples.obs['Disease'].isin(["NMO","Ageing","Uveitis"])].copy()

In [261]:
adata_pbmc_clean_samples.obs['Disease'].unique()

In [216]:
adata_csf_clean_samples = adata_csf_clean_samples[adata_csf_clean_samples.obs['cell_subtype_short'] != "ncMonocytes"].copy()

In [262]:
# 1. 先计算数据（可单独调试/保存数据）
data_dict_csf = calculate_roe_heatmap_data(
    adata=adata_csf_clean_samples, 
    age_col='age',  # 仅指定自定义的年龄列，其他用默认
    cell_type_col='cell_subtype_short'
)

data_dict_pbmc = calculate_roe_heatmap_data(
    adata=adata_pbmc_clean_samples, 
    age_col='age',  # 仅指定自定义的年龄列，其他用默认
    cell_type_col='cell_subtype_short'
)


In [222]:
# disease_order = [
#     "Healthy", "MS", "IIH", "Ageing", "MCI/AD", "PD",
#     "Ab-mediated IDD", "Neuro-COVID"
# ]

cell_order = [
    'Memory B','Plasma cell','Naive B','ILC2','pDC','ASDC','γδT','NKT','NK bright','CD8 Naive',
    'CD4 Memory','CD8 Memory','CD8 CTL','Treg','MAIT','CD4 CTL','CD4 Naive',
    'NK dim','Transit. NK','cMonocytes',
    'cDC2','trMac','infMac','mdMac'
]

plot_roe_heatmap1(
    data_dict_csf,
    cell_size=0.35,
    vmax=10,

    cell_order=cell_order,
    save_path="ROE_heatmap_fixed_disease_cell_order.pdf",
    cbar_x=0.77
)

In [197]:
type(data_dict_csf['roe_df'])

In [207]:
roe_df = data_dict_csf['roe_df']
roe_df_reset = roe_df.reset_index()
# 此时 cell_subtype_short 变成第一列，不再是行索引
roe_df.index

In [264]:
disease_order = [
    "Healthy", "MS", "IIH", "Ageing", 
    "Ab-mediated IDD", "CIS"
]


plot_roe_heatmap1(
    data_dict=data_dict_pbmc,
    cell_size=0.35,  # 单元格大小
    vmax=10,         # ROE色条最大值
    disease_order=disease_order,
    roe_colors = ["#244773", "#73A9D0", "#F8F8F8", "#D4887A", "#752C36"],  # 自定义ROE配色
    save_path='roe_heatmap_adata_pbmc.pdf',  # 保存路径（支持pdf/png/jpg等格式）
    dpi=300,  # 保存分辨率（300足够期刊使用）
    cbar_x=0.82   
)

In [278]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from scipy.stats import ranksums
from statsmodels.stats.multitest import multipletests
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize





# =============================================================================
# 0. 基本设置
# =============================================================================
REF_GROUP = "Healthy"

order_list = [
    "Healthy",
    "Ageing", "MCI/AD", "PD","IIH",
    "MS", "Ab-mediated IDD", "VE",
    "Neuro-COVID",  
    "CAR-T-DIPG","NT1",
]

broad_to_subtypes = {
    "T_cell": ["MAIT", "CD4 CTL", "NKT", "CD8 Naive", "CD4 Memory",
               "CD8 CTL", "CD8 Memory", "Treg", "γδT", "CD4 Naive"],
    "B_cell": ["Plasma cell", "Naive B", "Memory B"],
    "NK_cell": ["ILC2", "NK bright", "NK dim", "Transit. NK"],
    "Myeloid": ["cMonocytes", "cDC2", "infMac", "trMac",
                "ASDC", "mdMac", "ncMonocytes", "Basophil"]
}

manual_row_order = (
    broad_to_subtypes["B_cell"]
    + broad_to_subtypes["T_cell"]
    + broad_to_subtypes["NK_cell"]
    + broad_to_subtypes["Myeloid"]
)

outdir = "Figure3_and_SuppFigure2_custom_layout"
os.makedirs(outdir, exist_ok=True)

sns.set_style("white")


# =============================================================================
# 1. 工具函数
# =============================================================================
def ensure_categories(df, disease_col="Disease", order_list=None):
    present_order = [x for x in order_list if x in df[disease_col].astype(str).unique()]
    df[disease_col] = pd.Categorical(df[disease_col], categories=present_order, ordered=True)
    return df, present_order


def p_to_stars(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return ""


def bh_adjust_within_celltype(stats_df, p_col="p_value", group_col="cell_type"):
    stats_df = stats_df.copy()
    stats_df["p_adj"] = np.nan

    for ct, idx in stats_df.groupby(group_col).groups.items():
        pvals = stats_df.loc[idx, p_col].values.astype(float)
        valid_mask = np.isfinite(pvals)

        if valid_mask.sum() > 0:
            _, padj, _, _ = multipletests(pvals[valid_mask], method="fdr_bh")
            full = np.full(len(pvals), np.nan)
            full[valid_mask] = padj
            stats_df.loc[idx, "p_adj"] = full

    return stats_df


def get_global_absmax(effect_mats):
    vals = []
    for mat in effect_mats:
        arr = mat.values.astype(float).flatten()
        arr = arr[np.isfinite(arr)]
        vals.extend(arr.tolist())
    if len(vals) == 0:
        return 1.0
    return max(np.max(np.abs(vals)), 1.0)


def get_all_celltypes_order(observed_celltypes, manual_order):
    observed_celltypes = list(pd.unique(observed_celltypes))
    ordered = [x for x in manual_order if x in observed_celltypes]
    rest = [x for x in observed_celltypes if x not in ordered]
    return ordered + sorted(rest)

# =============================================================================
# 2. overall proportion
# =============================================================================
def calc_overall_fraction_df(adata):
    df = adata.obs[["orig.ident", "Disease", "cell_subtype_short"]].copy()

    count_df = (
        df.groupby(["orig.ident", "Disease", "cell_subtype_short"], observed=True)
          .size()
    )

    count_df = count_df.unstack(fill_value=0).stack()
    count_df = count_df.reset_index(name="n")

    sample_totals = (
        count_df.groupby("orig.ident", observed=False)["n"]
                .sum()
                .rename("sample_total")
                .reset_index()
    )

    frac_df = count_df.merge(sample_totals, on="orig.ident", how="left")
    frac_df["fraction"] = frac_df["n"] / frac_df["sample_total"] * 100.0

    frac_df, present_order = ensure_categories(frac_df, "Disease", order_list)
    return frac_df, present_order

# =============================================================================
# 3. within-lineage proportion
# =============================================================================
def calc_within_lineage_fraction_df(adata, lineage_name, subtype_list):
    df = adata.obs[["orig.ident", "Disease", "cell_subtype_short"]].copy()
    df = df[df["cell_subtype_short"].isin(subtype_list)].copy()

    count_df = (
        df.groupby(["orig.ident", "Disease", "cell_subtype_short"], observed=True)
          .size()
    )

    count_df = count_df.unstack(fill_value=0).stack()
    count_df = count_df.reset_index(name="n")

    lineage_totals = (
        count_df.groupby(["orig.ident", "Disease"], observed=False)["n"]
                .sum()
                .rename("lineage_total")
                .reset_index()
    )

    frac_df = count_df.merge(lineage_totals, on=["orig.ident", "Disease"], how="left")
    frac_df["fraction"] = frac_df["n"] / frac_df["lineage_total"] * 100.0
    frac_df["lineage"] = lineage_name

    frac_df, present_order = ensure_categories(frac_df, "Disease", order_list)
    return frac_df, present_order

# =============================================================================
# 4. 统计：Disease vs Healthy
# =============================================================================
def compare_vs_healthy(frac_df, ref_group="Healthy"):
    rows = []

    diseases = [d for d in frac_df["Disease"].cat.categories if d != ref_group]
    cell_types = list(frac_df["cell_subtype_short"].dropna().unique())

    for ct in cell_types:
        sub = frac_df[frac_df["cell_subtype_short"] == ct].copy()

        ref_vals = sub.loc[sub["Disease"] == ref_group, "fraction"].values
        ref_median = np.median(ref_vals) if len(ref_vals) > 0 else np.nan

        for dis in diseases:
            dis_vals = sub.loc[sub["Disease"] == dis, "fraction"].values
            dis_median = np.median(dis_vals) if len(dis_vals) > 0 else np.nan

            if len(ref_vals) >= 2 and len(dis_vals) >= 2:
                _, p = ranksums(ref_vals, dis_vals)
            else:
                p = np.nan

            effect = dis_median - ref_median if (
                np.isfinite(dis_median) and np.isfinite(ref_median)
            ) else np.nan

            rows.append({
                "cell_type": ct,
                "Disease": dis,
                "ref_group": ref_group,
                "ref_median": ref_median,
                "disease_median": dis_median,
                "median_diff": effect,
                "p_value": p
            })

    stats_df = pd.DataFrame(rows)
    stats_df = bh_adjust_within_celltype(stats_df, p_col="p_value", group_col="cell_type")
    stats_df["signif_raw"] = stats_df["p_value"].apply(p_to_stars)
    stats_df["signif_fdr"] = stats_df["p_adj"].apply(p_to_stars)
    return stats_df

# =============================================================================
# 5. 转 heatmap 矩阵
# =============================================================================
def stats_to_heatmap_mats(stats_df, row_order, col_order, signif_col="signif_raw"):
    effect_mat = (
        stats_df.pivot(index="cell_type", columns="Disease", values="median_diff")
                .reindex(index=row_order, columns=col_order)
    )

    star_mat = (
        stats_df.pivot(index="cell_type", columns="Disease", values=signif_col)
                .reindex(index=row_order, columns=col_order)
    )

    return effect_mat, star_mat

# =============================================================================
# 6. 构建 overall / lineage bundle
# =============================================================================
def build_overall_heatmap_bundle(adata):
    frac_df, present_order = calc_overall_fraction_df(adata)
    stats_df = compare_vs_healthy(frac_df, ref_group=REF_GROUP)

    disease_cols = [d for d in present_order if d != REF_GROUP]
    row_order = get_all_celltypes_order(
        observed_celltypes=stats_df["cell_type"].unique(),
        manual_order=manual_row_order
    )

    effect_raw, star_raw = stats_to_heatmap_mats(
        stats_df, row_order=row_order, col_order=disease_cols, signif_col="signif_raw"
    )
    effect_fdr, star_fdr = stats_to_heatmap_mats(
        stats_df, row_order=row_order, col_order=disease_cols, signif_col="signif_fdr"
    )

    return {
        "frac_df": frac_df,
        "stats_df": stats_df,
        "effect_raw": effect_raw,
        "star_raw": star_raw,
        "effect_fdr": effect_fdr,
        "star_fdr": star_fdr
    }


def build_lineage_heatmap_bundle(adata, lineage_name, subtype_list):
    frac_df, present_order = calc_within_lineage_fraction_df(adata, lineage_name, subtype_list)
    stats_df = compare_vs_healthy(frac_df, ref_group=REF_GROUP)

    disease_cols = [d for d in present_order if d != REF_GROUP]
    row_order = [x for x in subtype_list if x in stats_df["cell_type"].unique()]

    effect_raw, star_raw = stats_to_heatmap_mats(
        stats_df, row_order=row_order, col_order=disease_cols, signif_col="signif_raw"
    )
    effect_fdr, star_fdr = stats_to_heatmap_mats(
        stats_df, row_order=row_order, col_order=disease_cols, signif_col="signif_fdr"
    )

    return {
        "frac_df": frac_df,
        "stats_df": stats_df,
        "effect_raw": effect_raw,
        "star_raw": star_raw,
        "effect_fdr": effect_fdr,
        "star_fdr": star_fdr
    }






# =============================================================================
# 1. 自动计算 robust color scale（关键函数）
# =============================================================================
def get_robust_absmax(mat, q=0.98):
    vals = mat.values.flatten()
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return 1
    return np.quantile(np.abs(vals), q)


def get_global_absmax(mats, q=0.98):
    vals = np.concatenate([m.values.flatten() for m in mats])
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return 1
    return np.quantile(np.abs(vals), q)


# =============================================================================
# 2. 主绘图函数（最终版）
# =============================================================================
def plot_custom_A_BC_DE_layout(
    overall_item,
    sub_items,
    savepath,
    page_title="Figure",
    show_in_notebook=True,
    overall_absmax=None,
    sub_absmax=None,
    overall_cbar_label="Median difference vs Healthy (%)",
    sub_cbar_label="Median difference vs Healthy (%)"
):
    """
    overall_item: dict(title, effect_mat, star_mat)
    sub_items: list of 4 dict(title, effect_mat, star_mat)
    """

    # ========= 自动计算范围 =========
    if overall_absmax is None:
        overall_absmax = get_robust_absmax(overall_item["effect_mat"], q=0.98)

    if sub_absmax is None:
        sub_absmax = get_global_absmax(
            [x["effect_mat"] for x in sub_items], q=0.98
        )

    fig = plt.figure(figsize=(11.69, 8.27), dpi=300)

    # ========= layout =========
    gs = gridspec.GridSpec(
        nrows=20, ncols=24, figure=fig,
        left=0.04, right=0.86, top=0.91, bottom=0.08,
        wspace=1.55, hspace=1.95
    )

    # ========= 标题 =========
    fig.text(
        0.02, 0.965, page_title,
        fontsize=14, fontweight="bold", color="white",
        ha="left", va="top",
        bbox=dict(facecolor="black", edgecolor="black", boxstyle="square,pad=0.15")
    )

    # ========= A =========
    axA = fig.add_subplot(gs[3:18, 0:5])
    sns.heatmap(
        overall_item["effect_mat"],
        cmap="RdBu_r",
        center=0,
        vmin=-overall_absmax,
        vmax=overall_absmax,
        linewidths=0.22,
        linecolor="white",
        annot=overall_item["star_mat"],
        fmt="",
        annot_kws={"fontsize": 6, "fontweight": "bold"},
        cbar=False,
        ax=axA
    )
    axA.set_title(overall_item["title"], fontsize=9, pad=4)
    axA.set_xticklabels(axA.get_xticklabels(), rotation=45, ha="right", fontsize=5.5)
    axA.set_yticklabels(axA.get_yticklabels(), fontsize=5.5)
    axA.set_xlabel("")
    axA.set_ylabel("")
    # ========= B–E =========
    positions = [
        (3, 8, 10, 15),
        (3, 16, 10, 23),
        (12, 8, 16, 15),
        (12, 16, 16, 23)
    ]

    for i, item in enumerate(sub_items):
        r0, c0, r1, c1 = positions[i]
        ax = fig.add_subplot(gs[r0:r1, c0:c1])

        sns.heatmap(
            item["effect_mat"],
            cmap="RdBu_r",
            center=0,
            vmin=-sub_absmax,
            vmax=sub_absmax,
            linewidths=0.22,
            linecolor="white",
            annot=item["star_mat"],
            fmt="",
            annot_kws={"fontsize": 6.5, "fontweight": "bold"},
            cbar=False,
            ax=ax
        )

        ax.set_title(item["title"], fontsize=9, pad=4)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=5.5)
        ax.set_yticklabels(ax.get_yticklabels(), fontsize=5.5)
        ax.set_xlabel("")
        ax.set_ylabel("")
    # ========= Panel labels =========
    panel_pos = {
        "A": (0.045, 0.83),
        "B": (0.30, 0.83),
        "C": (0.565, 0.83),
        "D": (0.30, 0.43),
        "E": (0.565, 0.43),
    }

    for k, (x, y) in panel_pos.items():
        fig.text(x, y, k, fontsize=15, fontweight="bold")

    # ========= colorbars（高度统一） =========
    cbar_y = 0.29
    cbar_h = 0.39
    cbar_w = 0.008

    # A colorbar
    cbar_ax_A = fig.add_axes([0.207, cbar_y, cbar_w, cbar_h])
    smA = ScalarMappable(cmap="RdBu_r",
                         norm=Normalize(vmin=-overall_absmax, vmax=overall_absmax))
    smA.set_array([])
    cbarA = fig.colorbar(smA, cax=cbar_ax_A)
    cbarA.ax.set_title("A", fontsize=7, pad=4)
    cbarA.set_label(overall_cbar_label, fontsize=7)
    cbarA.ax.tick_params(labelsize=6)

    # B–E colorbar
    cbar_ax_sub = fig.add_axes([0.835, cbar_y, cbar_w, cbar_h])
    sm_sub = ScalarMappable(cmap="RdBu_r",
                            norm=Normalize(vmin=-sub_absmax, vmax=sub_absmax))
    sm_sub.set_array([])
    cbar_sub = fig.colorbar(sm_sub, cax=cbar_ax_sub)
    cbar_sub.ax.set_title("B–E", fontsize=7, pad=4)
    cbar_sub.set_label(sub_cbar_label, fontsize=7)
    cbar_sub.ax.tick_params(labelsize=6)

    # ========= 保存 =========
    plt.savefig(savepath, bbox_inches="tight")

    if show_in_notebook:
        plt.show()
    else:
        plt.close()

    print("✅ saved:", savepath)





# =============================================================================
# 8. 实际运行
# =============================================================================
overall_bundle = build_overall_heatmap_bundle(adata_csf_clean_samples)
t_bundle = build_lineage_heatmap_bundle(adata_csf_clean_samples, "T_cell", broad_to_subtypes["T_cell"])
b_bundle = build_lineage_heatmap_bundle(adata_csf_clean_samples, "B_cell", broad_to_subtypes["B_cell"])
nk_bundle = build_lineage_heatmap_bundle(adata_csf_clean_samples, "NK_cell", broad_to_subtypes["NK_cell"])
my_bundle = build_lineage_heatmap_bundle(adata_csf_clean_samples, "Myeloid", broad_to_subtypes["Myeloid"])

# 保存统计表
overall_bundle["stats_df"].to_csv(os.path.join(outdir, "overall_stats.csv"), index=False)
t_bundle["stats_df"].to_csv(os.path.join(outdir, "T_lineage_stats.csv"), index=False)
b_bundle["stats_df"].to_csv(os.path.join(outdir, "B_lineage_stats.csv"), index=False)
nk_bundle["stats_df"].to_csv(os.path.join(outdir, "NK_lineage_stats.csv"), index=False)
my_bundle["stats_df"].to_csv(os.path.join(outdir, "Myeloid_lineage_stats.csv"), index=False)

# B-E 统一色标，单独算
raw_sub_absmax = get_global_absmax([
    t_bundle["effect_raw"], my_bundle["effect_raw"], nk_bundle["effect_raw"], b_bundle["effect_raw"]
])

fdr_sub_absmax = get_global_absmax([
    t_bundle["effect_fdr"], my_bundle["effect_fdr"], nk_bundle["effect_fdr"], b_bundle["effect_fdr"]
])    

plot_custom_A_BC_DE_layout(
    overall_item={
        "title": "",
        "effect_mat": overall_bundle["effect_raw"],
        "star_mat": overall_bundle["star_raw"]
    },
    sub_items=[
        {"title": "", "effect_mat": t_bundle["effect_raw"], "star_mat": t_bundle["star_raw"]},
        {"title": "", "effect_mat": my_bundle["effect_raw"], "star_mat": my_bundle["star_raw"]},
        {"title": "", "effect_mat": nk_bundle["effect_raw"], "star_mat": nk_bundle["star_raw"]},
        {"title": "", "effect_mat": b_bundle["effect_raw"], "star_mat": b_bundle["star_raw"]},
    ],
    savepath="Figure3_final.pdf",
    page_title="Figure 3"
)


plot_custom_A_BC_DE_layout(
    overall_item={
        "title": "",
        "effect_mat": overall_bundle["effect_fdr"],
        "star_mat": overall_bundle["star_fdr"]
    },
    sub_items=[
        {"title": "", "effect_mat": t_bundle["effect_fdr"], "star_mat": t_bundle["star_fdr"]},
        {"title": "", "effect_mat": my_bundle["effect_fdr"], "star_mat": my_bundle["star_fdr"]},
        {"title": "", "effect_mat": nk_bundle["effect_fdr"], "star_mat": nk_bundle["star_fdr"]},
        {"title": "", "effect_mat": b_bundle["effect_fdr"], "star_mat": b_bundle["star_fdr"]},
    ],
    savepath="SupplementaryFigure2_final.pdf",
    page_title="Supplementary Figure 2"
)

In [245]:
adata_csf_clean_samples

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from scipy.stats import ranksums
from scipy.cluster.hierarchy import linkage, dendrogram
from statsmodels.stats.multitest import multipletests
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, LinearSegmentedColormap


# =============================================================================
# 0. 基本设置
# =============================================================================
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

sns.set_style("white")

REF_GROUP = "Healthy"

# =============================================================================
# 重要：
# 这里就是你想在图里显示的疾病顺序。
# 现在代码会严格按照这个顺序显示，不会自动追加其他疾病。
# 如果你想显示 Uveitis / CIS / RIS / NT2 / DLB / PHN，就把它们加到这里。
# =============================================================================
DEFAULT_DISEASE_ORDER = [
    "Ageing",
    "MCI/AD",
    "PD",
    "IIH",
    "MS",
    "Ab-mediated IDD",
    "VE",
    "Neuro-COVID",
    "CAR-T-DIPG",
    "NT1",
]

# 是否严格只保留 DEFAULT_DISEASE_ORDER 中的疾病
STRICT_KEEP_ONLY_DISEASE_ORDER = True

# 每组疾病最少样本数
MIN_SAMPLES_PER_DISEASE = 3

broad_to_subtypes = {
    "T_cell": [
        "MAIT", "CD4 CTL", "NKT", "CD8 Naive", "CD4 Memory",
        "CD8 CTL", "CD8 Memory", "Treg", "γδT", "CD4 Naive"
    ],
    "B_cell": [
        "Plasma cell", "Naive B", "Memory B", "Transitional B"
    ],
    "NK_cell": [
        "ILC2", "NK bright", "NK dim", "Transit. NK"
    ],
    "Myeloid": [
        "cMonocytes", "cDC2", "infMac", "trMac",
        "ASDC", "mdMac", "ncMonocytes", "Basophil"
    ]
}

manual_row_order = (
    broad_to_subtypes["B_cell"]
    + broad_to_subtypes["T_cell"]
    + broad_to_subtypes["NK_cell"]
    + broad_to_subtypes["Myeloid"]
)

outdir = "Figure3_and_SuppFigure2_custom_layout_strict_order"
os.makedirs(outdir, exist_ok=True)


# =============================================================================
# 1. 工具函数
# =============================================================================
def p_to_stars(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return ""


def ensure_categories(df, disease_col="Disease", order_list=None, strict=False):
    """
    strict=False:
        order_list 中的疾病排前面，其他 observed disease 追加到后面。

    strict=True:
        只保留 order_list 中出现的疾病，其他疾病直接去掉。
    """
    df = df.copy()

    observed = list(pd.unique(df[disease_col].dropna().astype(str)))

    if order_list is None:
        present_order = observed
    else:
        present_order = [x for x in order_list if x in observed]

        if not strict:
            extra = [x for x in observed if x not in present_order]
            present_order = present_order + sorted(extra)

        df = df[df[disease_col].astype(str).isin(present_order)].copy()

    df[disease_col] = pd.Categorical(
        df[disease_col].astype(str),
        categories=present_order,
        ordered=True
    )

    return df, present_order


def filter_diseases_by_min_samples(
    frac_df,
    min_samples=3,
    ref_group="Healthy",
    sample_col="orig.ident",
    disease_col="Disease"
):
    """
    只保留样本数 >= min_samples 的疾病。
    Healthy 也要求 >= min_samples，否则后续统计没有意义。
    """
    df = frac_df.copy()

    sample_meta = (
        df[[sample_col, disease_col]]
        .drop_duplicates()
        .dropna()
    )

    disease_n = (
        sample_meta
        .groupby(disease_col, observed=True)[sample_col]
        .nunique()
    )

    keep_diseases = disease_n[disease_n >= min_samples].index.astype(str).tolist()

    if ref_group not in keep_diseases:
        raise ValueError(
            f"{ref_group} has fewer than {min_samples} samples. "
            f"Please check sample filtering."
        )

    df = df[df[disease_col].astype(str).isin(keep_diseases)].copy()

    old_categories = list(df[disease_col].cat.categories) if hasattr(df[disease_col], "cat") else None

    if old_categories is not None:
        new_categories = [x for x in old_categories if x in keep_diseases]
        df[disease_col] = pd.Categorical(
            df[disease_col].astype(str),
            categories=new_categories,
            ordered=True
        )

    print("Disease sample number used:")
    print(disease_n.loc[keep_diseases].sort_index())

    return df, keep_diseases


def bh_adjust_within_celltype(stats_df, p_col="p_value", group_col="cell_type"):
    stats_df = stats_df.copy()
    stats_df["p_adj"] = np.nan

    for ct, idx in stats_df.groupby(group_col).groups.items():
        pvals = stats_df.loc[idx, p_col].values.astype(float)
        valid_mask = np.isfinite(pvals)

        if valid_mask.sum() > 0:
            _, padj, _, _ = multipletests(pvals[valid_mask], method="fdr_bh")

            full = np.full(len(pvals), np.nan)
            full[valid_mask] = padj

            stats_df.loc[idx, "p_adj"] = full

    return stats_df


def get_all_celltypes_order(observed_celltypes, manual_order):
    observed_celltypes = list(pd.unique(pd.Series(observed_celltypes).dropna().astype(str)))
    ordered = [x for x in manual_order if x in observed_celltypes]
    rest = [x for x in observed_celltypes if x not in ordered]
    return ordered + sorted(rest)


def get_robust_absmax(mat, q=0.98, min_absmax=1.0):
    vals = mat.values.astype(float).flatten()
    vals = vals[np.isfinite(vals)]

    if len(vals) == 0:
        return min_absmax

    return max(float(np.quantile(np.abs(vals), q)), min_absmax)


def get_global_absmax(mats, q=0.98, min_absmax=1.0):
    vals = []

    for mat in mats:
        arr = mat.values.astype(float).flatten()
        arr = arr[np.isfinite(arr)]

        if len(arr) > 0:
            vals.append(arr)

    if len(vals) == 0:
        return min_absmax

    vals = np.concatenate(vals)
    return max(float(np.quantile(np.abs(vals), q)), min_absmax)


def make_roe_cmap(
    roe_colors=("#244773", "#73A9D0", "#F8F8F8", "#D4887A", "#752C36")
):
    return LinearSegmentedColormap.from_list(
        "custom_RdBu",
        list(zip([0.0, 0.35, 0.5, 0.65, 1.0], roe_colors))
    )


def get_disease_cols_by_order_or_cluster(
    stats_df,
    present_order,
    ref_group="Healthy",
    disease_order=None,
    strict=True
):
    """
    disease_order=None:
        根据 median_diff profile 对疾病聚类，并返回 disease_Z。

    disease_order=list:
        按指定顺序排列，不返回 disease_Z。

    strict=True:
        只显示 disease_order 里面的疾病，不自动追加其他疾病。
    """
    observed_diseases = [
        d for d in present_order
        if d != ref_group and d in stats_df["Disease"].astype(str).unique()
    ]

    if len(observed_diseases) <= 1:
        return observed_diseases, None

    # -------------------------------------------------------------------------
    # 1. 指定疾病顺序
    # -------------------------------------------------------------------------
    if disease_order is not None:
        disease_order_no_ref = [
            d for d in disease_order
            if d != ref_group
        ]

        ordered = [
            d for d in disease_order_no_ref
            if d in observed_diseases
        ]

        missing = [
            d for d in disease_order_no_ref
            if d not in observed_diseases
        ]

        if missing:
            print("Warning: diseases in disease_order not found or filtered:", missing)

        if strict:
            return ordered, None

        extra = [
            d for d in observed_diseases
            if d not in ordered
        ]

        return ordered + extra, None

    # -------------------------------------------------------------------------
    # 2. 自动聚类疾病
    # -------------------------------------------------------------------------
    mat = (
        stats_df
        .pivot(index="cell_type", columns="Disease", values="median_diff")
        .reindex(columns=observed_diseases)
    )

    mat_for_cluster = mat.T.fillna(0)

    if mat_for_cluster.shape[0] <= 1:
        return mat_for_cluster.index.tolist(), None

    disease_Z = linkage(
        mat_for_cluster.values,
        method="ward",
        optimal_ordering=True
    )

    leaves = dendrogram(
        disease_Z,
        no_plot=True
    )["leaves"]

    clustered_diseases = mat_for_cluster.index[leaves].tolist()

    return clustered_diseases, disease_Z


# =============================================================================
# 2. Overall proportion
# =============================================================================
def calc_overall_fraction_df(
    adata,
    sample_col="orig.ident",
    disease_col="Disease",
    celltype_col="cell_subtype_short",
    disease_order=None,
    strict=False
):
    obs = adata.obs[[sample_col, disease_col, celltype_col]].copy()

    obs = obs.dropna(subset=[sample_col, disease_col, celltype_col])
    obs[sample_col] = obs[sample_col].astype(str)
    obs[disease_col] = obs[disease_col].astype(str)
    obs[celltype_col] = obs[celltype_col].astype(str)

    if disease_order is not None and strict:
        obs = obs[obs[disease_col].isin(disease_order + [REF_GROUP])].copy()

    sample_meta = (
        obs[[sample_col, disease_col]]
        .drop_duplicates(subset=[sample_col])
        .set_index(sample_col)
    )

    observed_celltypes = list(pd.unique(obs[celltype_col]))

    cell_order = get_all_celltypes_order(
        observed_celltypes=observed_celltypes,
        manual_order=manual_row_order
    )

    count_s = (
        obs.groupby([sample_col, celltype_col], observed=True)
           .size()
           .rename("n")
    )

    full_index = pd.MultiIndex.from_product(
        [sample_meta.index.tolist(), cell_order],
        names=[sample_col, celltype_col]
    )

    count_df = (
        count_s.reindex(full_index, fill_value=0)
               .reset_index()
               .merge(sample_meta.reset_index(), on=sample_col, how="left")
    )

    sample_total = count_df.groupby(sample_col)["n"].transform("sum")

    count_df["fraction"] = np.where(
        sample_total > 0,
        count_df["n"] / sample_total * 100.0,
        np.nan
    )

    count_df = count_df.rename(
        columns={
            sample_col: "orig.ident",
            disease_col: "Disease",
            celltype_col: "cell_subtype_short"
        }
    )

    order_with_ref = [REF_GROUP] + [d for d in disease_order if d != REF_GROUP] if disease_order is not None else None

    count_df, present_order = ensure_categories(
        count_df,
        disease_col="Disease",
        order_list=order_with_ref,
        strict=strict
    )

    return count_df, present_order


# =============================================================================
# 3. Within-lineage proportion
# =============================================================================
def calc_within_lineage_fraction_df(
    adata,
    lineage_name,
    subtype_list,
    sample_col="orig.ident",
    disease_col="Disease",
    celltype_col="cell_subtype_short",
    disease_order=None,
    strict=False
):
    obs = adata.obs[[sample_col, disease_col, celltype_col]].copy()

    obs = obs.dropna(subset=[sample_col, disease_col, celltype_col])
    obs[sample_col] = obs[sample_col].astype(str)
    obs[disease_col] = obs[disease_col].astype(str)
    obs[celltype_col] = obs[celltype_col].astype(str)

    if disease_order is not None and strict:
        obs = obs[obs[disease_col].isin(disease_order + [REF_GROUP])].copy()

    sample_meta = (
        obs[[sample_col, disease_col]]
        .drop_duplicates(subset=[sample_col])
        .set_index(sample_col)
    )

    subtype_list = [
        x for x in subtype_list
        if x in obs[celltype_col].unique()
    ]

    if len(subtype_list) == 0:
        raise ValueError(f"No valid subtypes found for lineage: {lineage_name}")

    obs_lineage = obs[obs[celltype_col].isin(subtype_list)].copy()

    count_s = (
        obs_lineage.groupby([sample_col, celltype_col], observed=True)
                   .size()
                   .rename("n")
    )

    full_index = pd.MultiIndex.from_product(
        [sample_meta.index.tolist(), subtype_list],
        names=[sample_col, celltype_col]
    )

    count_df = (
        count_s.reindex(full_index, fill_value=0)
               .reset_index()
               .merge(sample_meta.reset_index(), on=sample_col, how="left")
    )

    lineage_total = count_df.groupby(sample_col)["n"].transform("sum")

    count_df["fraction"] = np.where(
        lineage_total > 0,
        count_df["n"] / lineage_total * 100.0,
        np.nan
    )

    count_df["lineage"] = lineage_name

    count_df = count_df.rename(
        columns={
            sample_col: "orig.ident",
            disease_col: "Disease",
            celltype_col: "cell_subtype_short"
        }
    )

    order_with_ref = [REF_GROUP] + [d for d in disease_order if d != REF_GROUP] if disease_order is not None else None

    count_df, present_order = ensure_categories(
        count_df,
        disease_col="Disease",
        order_list=order_with_ref,
        strict=strict
    )

    return count_df, present_order


# =============================================================================
# 4. Disease vs Healthy 统计
# =============================================================================
def compare_vs_healthy(
    frac_df,
    ref_group="Healthy",
    min_samples=3
):
    rows = []

    diseases = [
        d for d in frac_df["Disease"].cat.categories
        if d != ref_group
    ]

    cell_types = list(
        pd.unique(frac_df["cell_subtype_short"].dropna().astype(str))
    )

    for ct in cell_types:
        sub = frac_df[frac_df["cell_subtype_short"].astype(str) == ct].copy()

        ref_vals = (
            sub.loc[sub["Disease"] == ref_group, "fraction"]
            .dropna()
            .values
        )

        ref_median = np.median(ref_vals) if len(ref_vals) > 0 else np.nan

        for dis in diseases:
            dis_vals = (
                sub.loc[sub["Disease"] == dis, "fraction"]
                .dropna()
                .values
            )

            dis_median = np.median(dis_vals) if len(dis_vals) > 0 else np.nan

            if len(ref_vals) >= min_samples and len(dis_vals) >= min_samples:
                _, p = ranksums(dis_vals, ref_vals)
            else:
                p = np.nan

            effect = (
                dis_median - ref_median
                if np.isfinite(dis_median) and np.isfinite(ref_median)
                else np.nan
            )

            rows.append({
                "cell_type": ct,
                "Disease": dis,
                "ref_group": ref_group,
                "n_ref": len(ref_vals),
                "n_disease": len(dis_vals),
                "ref_median": ref_median,
                "disease_median": dis_median,
                "median_diff": effect,
                "p_value": p
            })

    stats_df = pd.DataFrame(rows)

    stats_df = bh_adjust_within_celltype(
        stats_df,
        p_col="p_value",
        group_col="cell_type"
    )

    stats_df["signif_raw"] = stats_df["p_value"].apply(p_to_stars)
    stats_df["signif_fdr"] = stats_df["p_adj"].apply(p_to_stars)

    return stats_df


# =============================================================================
# 5. 转 heatmap 矩阵
# =============================================================================
def stats_to_heatmap_mats(
    stats_df,
    row_order,
    col_order,
    signif_col="signif_raw"
):
    effect_mat = (
        stats_df
        .pivot(index="cell_type", columns="Disease", values="median_diff")
        .reindex(index=row_order, columns=col_order)
    )

    star_mat = (
        stats_df
        .pivot(index="cell_type", columns="Disease", values=signif_col)
        .reindex(index=row_order, columns=col_order)
        .fillna("")
    )

    return effect_mat, star_mat


# =============================================================================
# 6. 构建 overall / lineage bundle
# =============================================================================
def build_overall_heatmap_bundle(
    adata,
    disease_order=None,
    min_samples=3,
    strict=True
):
    frac_df, present_order = calc_overall_fraction_df(
        adata,
        disease_order=disease_order,
        strict=strict
    )

    frac_df, keep_diseases = filter_diseases_by_min_samples(
        frac_df,
        min_samples=min_samples,
        ref_group=REF_GROUP
    )

    # 重新整理 categories，保证过滤后顺序仍然正确
    order_with_ref = [REF_GROUP] + [d for d in disease_order if d != REF_GROUP] if disease_order is not None else None
    frac_df, present_order = ensure_categories(
        frac_df,
        disease_col="Disease",
        order_list=order_with_ref,
        strict=strict
    )

    stats_df = compare_vs_healthy(
        frac_df,
        ref_group=REF_GROUP,
        min_samples=min_samples
    )

    disease_cols, disease_Z = get_disease_cols_by_order_or_cluster(
        stats_df=stats_df,
        present_order=present_order,
        ref_group=REF_GROUP,
        disease_order=disease_order,
        strict=strict
    )

    row_order = get_all_celltypes_order(
        observed_celltypes=stats_df["cell_type"].unique(),
        manual_order=manual_row_order
    )

    effect_raw, star_raw = stats_to_heatmap_mats(
        stats_df,
        row_order=row_order,
        col_order=disease_cols,
        signif_col="signif_raw"
    )

    effect_fdr, star_fdr = stats_to_heatmap_mats(
        stats_df,
        row_order=row_order,
        col_order=disease_cols,
        signif_col="signif_fdr"
    )

    return {
        "frac_df": frac_df,
        "stats_df": stats_df,
        "disease_cols": disease_cols,
        "disease_Z": disease_Z,
        "effect_raw": effect_raw,
        "star_raw": star_raw,
        "effect_fdr": effect_fdr,
        "star_fdr": star_fdr
    }


def build_lineage_heatmap_bundle(
    adata,
    lineage_name,
    subtype_list,
    disease_order=None,
    min_samples=3,
    strict=True
):
    frac_df, present_order = calc_within_lineage_fraction_df(
        adata,
        lineage_name=lineage_name,
        subtype_list=subtype_list,
        disease_order=disease_order,
        strict=strict
    )

    frac_df, keep_diseases = filter_diseases_by_min_samples(
        frac_df,
        min_samples=min_samples,
        ref_group=REF_GROUP
    )

    order_with_ref = [REF_GROUP] + [d for d in disease_order if d != REF_GROUP] if disease_order is not None else None
    frac_df, present_order = ensure_categories(
        frac_df,
        disease_col="Disease",
        order_list=order_with_ref,
        strict=strict
    )

    stats_df = compare_vs_healthy(
        frac_df,
        ref_group=REF_GROUP,
        min_samples=min_samples
    )

    disease_cols, disease_Z = get_disease_cols_by_order_or_cluster(
        stats_df=stats_df,
        present_order=present_order,
        ref_group=REF_GROUP,
        disease_order=disease_order,
        strict=strict
    )

    row_order = [
        x for x in subtype_list
        if x in stats_df["cell_type"].unique()
    ]

    effect_raw, star_raw = stats_to_heatmap_mats(
        stats_df,
        row_order=row_order,
        col_order=disease_cols,
        signif_col="signif_raw"
    )

    effect_fdr, star_fdr = stats_to_heatmap_mats(
        stats_df,
        row_order=row_order,
        col_order=disease_cols,
        signif_col="signif_fdr"
    )

    return {
        "frac_df": frac_df,
        "stats_df": stats_df,
        "disease_cols": disease_cols,
        "disease_Z": disease_Z,
        "effect_raw": effect_raw,
        "star_raw": star_raw,
        "effect_fdr": effect_fdr,
        "star_fdr": star_fdr
    }


# =============================================================================
# 7. 绘图函数
# =============================================================================
def draw_heatmap_panel(
    ax,
    effect_mat,
    star_mat,
    absmax,
    cmap,
    xtick_fontsize=5.5,
    ytick_fontsize=5.5,
    annot_fontsize=6,
    rasterized=False
):
    sns.heatmap(
        effect_mat,
        cmap=cmap,
        center=0,
        vmin=-absmax,
        vmax=absmax,
        linewidths=0.22,
        linecolor="white",
        annot=star_mat.fillna(""),
        fmt="",
        annot_kws={
            "fontsize": annot_fontsize,
            "fontweight": "bold"
        },
        cbar=False,
        ax=ax
    )

    if rasterized:
        for coll in ax.collections:
            coll.set_rasterized(True)

    ax.set_xlabel("")
    ax.set_ylabel("")

    ax.set_xticklabels(
        ax.get_xticklabels(),
        rotation=45,
        ha="right",
        fontsize=xtick_fontsize
    )

    ax.set_yticklabels(
        ax.get_yticklabels(),
        rotation=0,
        fontsize=ytick_fontsize
    )


def draw_column_dendrogram_above(
    fig,
    ax,
    Z,
    n_cols,
    height=0.035,
    pad=0.006,
    color="#333333",
    lw=0.6
):
    if Z is None or n_cols <= 1:
        return None

    pos = ax.get_position()

    ax_dend = fig.add_axes([
        pos.x0,
        pos.y1 + pad,
        pos.width,
        height
    ])

    dendro = dendrogram(
        Z,
        no_plot=True
    )

    max_y = 0

    for xs, ys in zip(dendro["icoord"], dendro["dcoord"]):
        xs = np.asarray(xs) / 10.0
        ys = np.asarray(ys)

        ax_dend.plot(
            xs,
            ys,
            color=color,
            lw=lw
        )

        max_y = max(max_y, np.max(ys))

    ax_dend.set_xlim(0, n_cols)
    ax_dend.set_ylim(0, max_y * 1.05)
    ax_dend.axis("off")

    return ax_dend


def plot_custom_A_BC_DE_layout(
    overall_item,
    sub_items,
    savepath,
    page_title="Figure",
    show_in_notebook=True,
    overall_absmax=None,
    sub_absmax=None,
    overall_cbar_label="Median difference vs Healthy (%)",
    sub_cbar_label="Median difference vs Healthy (%)",
    roe_colors=("#244773", "#73A9D0", "#F8F8F8", "#D4887A", "#752C36"),
    dpi=300,
    rasterized=False,
    disease_Z=None,
    show_disease_dendrogram=False,
    dendrogram_for_all_panels=False
):
    cmap = make_roe_cmap(roe_colors)

    if overall_absmax is None:
        overall_absmax = get_robust_absmax(
            overall_item["effect_mat"],
            q=0.98
        )

    if sub_absmax is None:
        sub_absmax = get_global_absmax(
            [x["effect_mat"] for x in sub_items],
            q=0.98
        )

    fig = plt.figure(figsize=(11.69, 8.27), dpi=dpi)

    gs = gridspec.GridSpec(
        nrows=20,
        ncols=24,
        figure=fig,
        left=0.04,
        right=0.86,
        top=0.88,
        bottom=0.08,
        wspace=1.55,
        hspace=1.95
    )

    fig.text(
        0.02,
        0.965,
        page_title,
        fontsize=14,
        fontweight="bold",
        color="white",
        ha="left",
        va="top",
        bbox=dict(
            facecolor="black",
            edgecolor="black",
            boxstyle="square,pad=0.15"
        )
    )

    # -------------------------------------------------------------------------
    # A panel
    # -------------------------------------------------------------------------
    axA = fig.add_subplot(gs[3:18, 0:5])

    draw_heatmap_panel(
        axA,
        overall_item["effect_mat"],
        overall_item["star_mat"],
        absmax=overall_absmax,
        cmap=cmap,
        annot_fontsize=6,
        rasterized=rasterized
    )

    axA.set_title(
        overall_item.get("title", ""),
        fontsize=9,
        pad=4
    )

    if show_disease_dendrogram and disease_Z is not None:
        draw_column_dendrogram_above(
            fig=fig,
            ax=axA,
            Z=disease_Z,
            n_cols=overall_item["effect_mat"].shape[1],
            height=0.040,
            pad=0.008,
            color="#333333",
            lw=0.6
        )

    # -------------------------------------------------------------------------
    # B-E panels
    # -------------------------------------------------------------------------
    positions = [
        (3, 8, 10, 15),
        (3, 16, 10, 23),
        (12, 8, 16, 15),
        (12, 16, 16, 23)
    ]

    for i, item in enumerate(sub_items):
        r0, c0, r1, c1 = positions[i]

        ax = fig.add_subplot(gs[r0:r1, c0:c1])

        draw_heatmap_panel(
            ax,
            item["effect_mat"],
            item["star_mat"],
            absmax=sub_absmax,
            cmap=cmap,
            annot_fontsize=6.5,
            rasterized=rasterized
        )

        ax.set_title(
            item.get("title", ""),
            fontsize=9,
            pad=4
        )

        if (
            show_disease_dendrogram
            and dendrogram_for_all_panels
            and disease_Z is not None
        ):
            draw_column_dendrogram_above(
                fig=fig,
                ax=ax,
                Z=disease_Z,
                n_cols=item["effect_mat"].shape[1],
                height=0.030,
                pad=0.006,
                color="#333333",
                lw=0.55
            )

    # -------------------------------------------------------------------------
    # Panel labels
    # -------------------------------------------------------------------------
    panel_pos = {
        "A": (0.045, 0.83),
        "B": (0.30, 0.83),
        "C": (0.565, 0.83),
        "D": (0.30, 0.43),
        "E": (0.565, 0.43),
    }

    for k, (x, y) in panel_pos.items():
        fig.text(
            x,
            y,
            k,
            fontsize=15,
            fontweight="bold"
        )

    # -------------------------------------------------------------------------
    # Colorbars
    # -------------------------------------------------------------------------
    cbar_y = 0.29
    cbar_h = 0.39
    cbar_w = 0.008

    cbar_ax_A = fig.add_axes([0.207, cbar_y, cbar_w, cbar_h])

    smA = ScalarMappable(
        cmap=cmap,
        norm=Normalize(
            vmin=-overall_absmax,
            vmax=overall_absmax
        )
    )

    smA.set_array([])

    cbarA = fig.colorbar(
        smA,
        cax=cbar_ax_A
    )

    cbarA.ax.set_title(
        "A",
        fontsize=7,
        pad=4
    )

    cbarA.set_label(
        overall_cbar_label,
        fontsize=7
    )

    cbarA.ax.tick_params(
        labelsize=6,
        length=0
    )

    cbarA.outline.set_visible(False)

    cbar_ax_sub = fig.add_axes([0.835, cbar_y, cbar_w, cbar_h])

    sm_sub = ScalarMappable(
        cmap=cmap,
        norm=Normalize(
            vmin=-sub_absmax,
            vmax=sub_absmax
        )
    )

    sm_sub.set_array([])

    cbar_sub = fig.colorbar(
        sm_sub,
        cax=cbar_ax_sub
    )

    cbar_sub.ax.set_title(
        "B-E",
        fontsize=7,
        pad=4
    )

    cbar_sub.set_label(
        sub_cbar_label,
        fontsize=7
    )

    cbar_sub.ax.tick_params(
        labelsize=6,
        length=0
    )

    cbar_sub.outline.set_visible(False)

    # -------------------------------------------------------------------------
    # Save
    # -------------------------------------------------------------------------
    fig.savefig(
        savepath,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="none"
    )

    if show_in_notebook:
        plt.show()
    else:
        plt.close(fig)

    print("✅ saved:", savepath)


# =============================================================================
# 8. 实际运行
# =============================================================================

# 手动指定疾病顺序
disease_order_for_plot = DEFAULT_DISEASE_ORDER

# 如果想自动聚类疾病，就改成：
# disease_order_for_plot = None

overall_bundle = build_overall_heatmap_bundle(
    adata_csf_clean_samples,
    disease_order=disease_order_for_plot,
    min_samples=MIN_SAMPLES_PER_DISEASE,
    strict=STRICT_KEEP_ONLY_DISEASE_ORDER
)

# 关键：
# 不再直接使用 DEFAULT_DISEASE_ORDER，而是使用 overall_bundle 里面实际保留后的列顺序。
final_disease_order_for_panels = overall_bundle["disease_cols"]

print("\nFinal disease columns actually used in all panels:")
print(final_disease_order_for_panels)

t_bundle = build_lineage_heatmap_bundle(
    adata_csf_clean_samples,
    "T_cell",
    broad_to_subtypes["T_cell"],
    disease_order=final_disease_order_for_panels,
    min_samples=MIN_SAMPLES_PER_DISEASE,
    strict=True
)

my_bundle = build_lineage_heatmap_bundle(
    adata_csf_clean_samples,
    "Myeloid",
    broad_to_subtypes["Myeloid"],
    disease_order=final_disease_order_for_panels,
    min_samples=MIN_SAMPLES_PER_DISEASE,
    strict=True
)

nk_bundle = build_lineage_heatmap_bundle(
    adata_csf_clean_samples,
    "NK_cell",
    broad_to_subtypes["NK_cell"],
    disease_order=final_disease_order_for_panels,
    min_samples=MIN_SAMPLES_PER_DISEASE,
    strict=True
)

b_bundle = build_lineage_heatmap_bundle(
    adata_csf_clean_samples,
    "B_cell",
    broad_to_subtypes["B_cell"],
    disease_order=final_disease_order_for_panels,
    min_samples=MIN_SAMPLES_PER_DISEASE,
    strict=True
)

print("\nCheck actual matrix columns:")
print("Overall:", overall_bundle["effect_raw"].columns.tolist())
print("T      :", t_bundle["effect_raw"].columns.tolist())
print("Myeloid:", my_bundle["effect_raw"].columns.tolist())
print("NK     :", nk_bundle["effect_raw"].columns.tolist())
print("B      :", b_bundle["effect_raw"].columns.tolist())


# =============================================================================
# 9. 保存统计表
# =============================================================================
overall_bundle["stats_df"].to_csv(
    os.path.join(outdir, "overall_stats.csv"),
    index=False
)

t_bundle["stats_df"].to_csv(
    os.path.join(outdir, "T_lineage_stats.csv"),
    index=False
)

my_bundle["stats_df"].to_csv(
    os.path.join(outdir, "Myeloid_lineage_stats.csv"),
    index=False
)

nk_bundle["stats_df"].to_csv(
    os.path.join(outdir, "NK_lineage_stats.csv"),
    index=False
)

b_bundle["stats_df"].to_csv(
    os.path.join(outdir, "B_lineage_stats.csv"),
    index=False
)


# =============================================================================
# 10. 统一色标
# =============================================================================
overall_absmax = get_robust_absmax(
    overall_bundle["effect_raw"],
    q=0.98
)

sub_absmax = get_global_absmax(
    [
        t_bundle["effect_raw"],
        my_bundle["effect_raw"],
        nk_bundle["effect_raw"],
        b_bundle["effect_raw"]
    ],
    q=0.98
)


# =============================================================================
# 11. Figure 3: raw P value stars
# =============================================================================
plot_custom_A_BC_DE_layout(
    overall_item={
        "title": "",
        "effect_mat": overall_bundle["effect_raw"],
        "star_mat": overall_bundle["star_raw"]
    },
    sub_items=[
        {
            "title": "",
            "effect_mat": t_bundle["effect_raw"],
            "star_mat": t_bundle["star_raw"]
        },
        {
            "title": "",
            "effect_mat": my_bundle["effect_raw"],
            "star_mat": my_bundle["star_raw"]
        },
        {
            "title": "",
            "effect_mat": nk_bundle["effect_raw"],
            "star_mat": nk_bundle["star_raw"]
        },
        {
            "title": "",
            "effect_mat": b_bundle["effect_raw"],
            "star_mat": b_bundle["star_raw"]
        },
    ],
    overall_absmax=overall_absmax,
    sub_absmax=sub_absmax,
    savepath=os.path.join(outdir, "Figure3_final.pdf"),
    page_title="Figure 3",
    rasterized=False,
    disease_Z=None,
    show_disease_dendrogram=False,
    dendrogram_for_all_panels=False
)


# =============================================================================
# 12. Supplementary Figure 2: FDR stars
# =============================================================================
plot_custom_A_BC_DE_layout(
    overall_item={
        "title": "",
        "effect_mat": overall_bundle["effect_fdr"],
        "star_mat": overall_bundle["star_fdr"]
    },
    sub_items=[
        {
            "title": "",
            "effect_mat": t_bundle["effect_fdr"],
            "star_mat": t_bundle["star_fdr"]
        },
        {
            "title": "",
            "effect_mat": my_bundle["effect_fdr"],
            "star_mat": my_bundle["star_fdr"]
        },
        {
            "title": "",
            "effect_mat": nk_bundle["effect_fdr"],
            "star_mat": nk_bundle["star_fdr"]
        },
        {
            "title": "",
            "effect_mat": b_bundle["effect_fdr"],
            "star_mat": b_bundle["star_fdr"]
        },
    ],
    overall_absmax=overall_absmax,
    sub_absmax=sub_absmax,
    savepath=os.path.join(outdir, "SupplementaryFigure2_final.pdf"),
    page_title="Supplementary Figure 2",
    rasterized=False,
    disease_Z=None,
    show_disease_dendrogram=False,
    dendrogram_for_all_panels=False
)

In [247]:
import scanpy as sc
import pandas as pd
from scipy.io import mmwrite
from scipy.sparse import issparse

adata = adata_csf_clean_samples

# 原始count矩阵：adata是 细胞(行) × 基因(列)，Seurat需要 基因(行) × 细胞(列)
count_mat = adata.layers["counts"].T  # 转置！关键修复
gene_names = adata.var_names
cell_names = adata.obs_names

# 导出稀疏矩阵 + 基因/细胞名
mmwrite("raw_counts.mtx", count_mat)
pd.DataFrame(gene_names).to_csv("genes.tsv", sep="\t", header=None, index=None)
pd.DataFrame(cell_names).to_csv("barcodes.tsv", sep="\t", header=None, index=None)

# 导出metadata
adata.obs.to_csv("metadata.csv", index=True)

# 导出UMAP坐标
umap_df = pd.DataFrame(
    adata.obsm["X_umap"],
    index=adata.obs_names,
    columns=["UMAP_1", "UMAP_2"]
)
umap_df.to_csv("umap_coords.csv", index=True)

print("已重新导出，矩阵完成转置，适配Seurat行列要求")
print(f"矩阵维度(基因×细胞): {count_mat.shape}")
print(f"基因数量: {len(gene_names)}, 细胞数量: {len(cell_names)}")

In [281]:

adata = sc.read_h5ad("/home/lishaogang/NEU2026_May/GEO_data/adata_final_0610.h5ad")

In [282]:
adata.obs['cell_type'].unique()

In [4]:
adata_Myeloid  = adata[adata.obs['cell_type'].isin(["Myeloid cell","pDC"])].copy()

In [5]:
adata_Myeloid

In [7]:
import scanpy as sc
import scanpy.external as sce
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt

# ============================================================
# 0. 复制对象，避免覆盖原始 adata_Myeloid
# ============================================================

adata_myeloid_log = adata_Myeloid.copy()

print(adata_myeloid_log)

# ============================================================
# 1. 使用 logcounts 作为表达矩阵
#    logcounts 通常已经是 log-normalized data，不要再 Normalize / log1p
# ============================================================

if "logcounts" not in adata_myeloid_log.layers:
    raise ValueError("❌ adata_Myeloid.layers 中没有 'logcounts'")

adata_myeloid_log.X = adata_myeloid_log.layers["logcounts"].copy()

print("✅ 已将 X 替换为 layers['logcounts']")

# ============================================================
# 2. 检查 dataset 列
# ============================================================

batch_key = "dataset"

if batch_key not in adata_myeloid_log.obs.columns:
    raise ValueError(f"❌ obs 中没有 '{batch_key}' 列，无法进行 Harmony 整合")

adata_myeloid_log.obs[batch_key] = (
    adata_myeloid_log.obs[batch_key]
    .astype(str)
    .astype("category")
)

print("Dataset distribution:")
print(adata_myeloid_log.obs[batch_key].value_counts())

# ============================================================
# 3. 清理异常值，防止 PCA / Harmony 报错
# ============================================================

if sp.issparse(adata_myeloid_log.X):
    adata_myeloid_log.X.data = np.nan_to_num(
        adata_myeloid_log.X.data,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )
else:
    adata_myeloid_log.X = np.nan_to_num(
        adata_myeloid_log.X,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

# ============================================================
# 4. 重新筛选 Myeloid 内部高变基因
# ============================================================

sc.pp.highly_variable_genes(
    adata_myeloid_log,
    n_top_genes=3000,
    flavor="seurat"
)

print("HVG number:", adata_myeloid_log.var["highly_variable"].sum())

# 只用 HVG 做降维聚类
adata_myeloid_hvg = adata_myeloid_log[:, adata_myeloid_log.var["highly_variable"]].copy()

print(adata_myeloid_hvg)

# ============================================================
# 5. Scale
# ============================================================

sc.pp.scale(
    adata_myeloid_hvg,
    max_value=10
)

# ============================================================
# 6. PCA
# ============================================================

sc.tl.pca(
    adata_myeloid_hvg,
    n_comps=50,
    svd_solver="arpack"
)

sc.pl.pca_variance_ratio(
    adata_myeloid_hvg,
    n_pcs=50,
    log=True
)

# ============================================================
# 7. Harmony 整合 dataset
#    关键步骤：基于 PCA 空间校正 dataset batch effect
# ============================================================

sce.pp.harmony_integrate(
    adata_myeloid_hvg,
    key=batch_key,
    basis="X_pca",
    adjusted_basis="X_pca_harmony_logcounts",
    max_iter_harmony=30
)

print("✅ Harmony finished")
print(adata_myeloid_hvg.obsm.keys())

# ============================================================
# 8. 基于 Harmony 后的 PCA 空间构建邻接图
# ============================================================

sc.pp.neighbors(
    adata_myeloid_hvg,
    n_neighbors=30,
    n_pcs=30,
    use_rep="X_pca_harmony_logcounts",
    metric="cosine"
)

# ============================================================
# 9. Leiden 聚类
# ============================================================

sc.tl.leiden(
    adata_myeloid_hvg,
    resolution=0.6,
    key_added="leiden_logcounts_harmony"
)

print("Cluster distribution:")
print(
    adata_myeloid_hvg.obs["leiden_logcounts_harmony"]
    .value_counts()
    .sort_index()
)

# ============================================================
# 10. UMAP
#    注意：这里 UMAP 是基于 Harmony 后的 neighbors 得到的
# ============================================================

sc.tl.umap(
    adata_myeloid_hvg,
    min_dist=0.35,
    spread=1.0
)

# ============================================================
# 11. 绘制 UMAP
# ============================================================

sc.pl.umap(
    adata_myeloid_hvg,
    color=[
        "leiden_logcounts_harmony",
        "cell_subtype_short",
        "Disease",
        "dataset"
    ],
    ncols=2,
    frameon=False,
    size=8,
    legend_loc="right margin"
)

# 聚类标签放图上
sc.pl.umap(
    adata_myeloid_hvg,
    color="leiden_logcounts_harmony",
    frameon=False,
    size=8,
    legend_loc="on data",
    legend_fontsize=8,
    legend_fontoutline=2,
    title="Myeloid reclustering based on logcounts + Harmony(dataset)"
)

# ============================================================
# 12. 保存重新聚类后的对象
# ============================================================

adata_myeloid_hvg.write_h5ad(
    "adata_Myeloid_logcounts_Harmony_dataset_reclustered.h5ad"
)

print("✅ 已保存: adata_Myeloid_logcounts_Harmony_dataset_reclustered.h5ad")

In [8]:
# ============================================================
# 把 Harmony 后的聚类 / UMAP / PCA 写回原始 adata_Myeloid
# ============================================================

adata_Myeloid.obs["leiden_logcounts_harmony"] = (
    adata_myeloid_hvg.obs["leiden_logcounts_harmony"].copy()
)

adata_Myeloid.obsm["X_umap_logcounts_harmony"] = (
    adata_myeloid_hvg.obsm["X_umap"].copy()
)

adata_Myeloid.obsm["X_pca_logcounts"] = (
    adata_myeloid_hvg.obsm["X_pca"].copy()
)

adata_Myeloid.obsm["X_pca_harmony_logcounts"] = (
    adata_myeloid_hvg.obsm["X_pca_harmony_logcounts"].copy()
)

adata_Myeloid.uns["leiden_logcounts_harmony"] = (
    adata_myeloid_hvg.uns["leiden_logcounts_harmony"].copy()
)

adata_Myeloid.uns["umap_logcounts_harmony"] = (
    adata_myeloid_hvg.uns["umap"].copy()
)

print("✅ 已写回 adata_Myeloid")
print("obs:  leiden_logcounts_harmony")
print("obsm: X_umap_logcounts_harmony")
print("obsm: X_pca_logcounts")
print("obsm: X_pca_harmony_logcounts")

In [9]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from adjustText import adjust_text

# ====================== 全局绘图参数（SCI Arial字体规范） ======================
celltype_col = "cell_subtype_short"

# 关键修改：使用 Harmony + logcounts 后的新 UMAP
basis = "X_umap_logcounts_harmony"

arial_family = "Arial"

mpl.rcParams["font.family"] = arial_family
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

# ====================== 高区分度配色 ======================
soft_color_map = {
    "cMonocytes": "#c84242",
    "ncMonocytes": "#d6cd9c",
    "mdMac": "#b97cb9",
    "trMac": "#d16565",
    "infMac": "#7b5bb8",
    "cDC2": "#967262",
    "pDC": "#7096c7",
    "ASDC": "#3675b9"
}

# ====================== 数据准备 ======================
adata_plot = adata_Myeloid.copy()

# 检查新 UMAP 是否存在
if basis not in adata_plot.obsm.keys():
    raise ValueError(
        f"❌ adata_Myeloid.obsm 中没有 {basis}。\n"
        f"请先确认你是否已经把 Harmony 后的 UMAP 写回 adata_Myeloid：\n"
        f"adata_Myeloid.obsm['X_umap_logcounts_harmony'] = adata_myeloid_hvg.obsm['X_umap'].copy()"
    )

# ====================== 改名 maMac → mdMac ======================
ser = adata_plot.obs[celltype_col]

if ser.dtype.name == "category":
    new_cats = {
        cat: "mdMac" if cat == "maMac" else cat
        for cat in ser.cat.categories
    }
    adata_plot.obs[celltype_col] = ser.cat.rename_categories(new_cats)
else:
    adata_plot.obs[celltype_col] = ser.replace("maMac", "mdMac")

# ====================== 剔除 Basophil ======================
keep_mask = adata_plot.obs[celltype_col] != "Basophil"
adata_plot = adata_plot[keep_mask, :].copy()

# 去掉未使用的 category，避免颜色和图例错位
if adata_plot.obs[celltype_col].dtype.name == "category":
    adata_plot.obs[celltype_col] = (
        adata_plot.obs[celltype_col]
        .cat.remove_unused_categories()
    )

# 固定细胞类型顺序
celltype_order = [
    "cMonocytes",
    "ncMonocytes",
    "trMac",
    "infMac",
    "mdMac",
    "cDC2",
    "pDC",
    "ASDC"
]

celltype_order = [
    ct for ct in celltype_order
    if ct in adata_plot.obs[celltype_col].astype(str).unique()
]

adata_plot.obs[celltype_col] = pd.Categorical(
    adata_plot.obs[celltype_col].astype(str),
    categories=celltype_order,
    ordered=True
)

# ====================== 自动匹配颜色 ======================
missing_colors = [
    ct for ct in celltype_order
    if ct not in soft_color_map
]

if missing_colors:
    print("⚠️ 缺失颜色定义细胞：", missing_colors)

adata_plot.uns[f"{celltype_col}_colors"] = [
    soft_color_map.get(ct, "#BDBDBD")
    for ct in celltype_order
]

# ====================== 计算每个细胞群中心点 ======================
def get_group_center(adata, groupby, basis="X_umap"):
    df_coord = pd.DataFrame(
        adata.obsm[basis],
        columns=["UMAP1", "UMAP2"],
        index=adata.obs_names
    )

    df_coord[groupby] = adata.obs[groupby].astype(str).values

    center_dict = {}

    for group, sub_df in df_coord.groupby(groupby, observed=True):
        # 用 median 比 mean 更稳，避免被离群点拉偏
        center_dict[group] = sub_df[["UMAP1", "UMAP2"]].median(axis=0).values

    return center_dict


label_center = get_group_center(
    adata_plot,
    groupby=celltype_col,
    basis=basis
)

# ====================== 创建画布绘图 ======================
fig, ax = plt.subplots(figsize=(8, 8))

# 关键修改：这里不能再用 sc.pl.umap
# 因为 sc.pl.umap 默认只画 adata.obsm["X_umap"]
sc.pl.embedding(
    adata_plot,
    basis=basis,
    color=celltype_col,
    size=8,
    alpha=1,
    ax=ax,
    show=False,
    frameon=False,
    legend_loc=None,
    title=None
)

# 散点光栅化
for patch in ax.collections:
    patch.set_rasterized(True)

# ====================== 带白色描边文字标签 ======================
text_list = []

for cell_type in celltype_order:

    if cell_type not in label_center:
        continue

    x_pos, y_pos = label_center[cell_type]

    txt = ax.text(
        x_pos,
        y_pos,
        cell_type,
        fontsize=11,
        fontfamily=arial_family,
        weight="bold",
        ha="center",
        va="center",
        color="black",
        path_effects=[
            pe.withStroke(
                linewidth=3,
                foreground="white"
            )
        ]
    )

    text_list.append(txt)

# 自动避让标签
adjust_text(
    text_list,
    ax=ax,
    expand_text=(1.1, 1.2),
    force_text=(0.3, 0.4),
    arrowprops=dict(
        arrowstyle="-",
        lw=0.7,
        color="#333333",
        alpha=0.5
    )
)

# ====================== 自定义 UMAP 坐标轴箭头 ======================
arrow_pos_x = 0.05
arrow_pos_y = 0.05
arrow_length = 0.16

# 横向 UMAP1 箭头
ax.annotate(
    "",
    xy=(arrow_pos_x + arrow_length, arrow_pos_y),
    xytext=(arrow_pos_x, arrow_pos_y),
    xycoords="axes fraction",
    arrowprops=dict(
        arrowstyle="->",
        lw=2,
        color="black"
    )
)

# 纵向 UMAP2 箭头
ax.annotate(
    "",
    xy=(arrow_pos_x, arrow_pos_y + arrow_length),
    xytext=(arrow_pos_x, arrow_pos_y),
    xycoords="axes fraction",
    arrowprops=dict(
        arrowstyle="->",
        lw=2,
        color="black"
    )
)

# UMAP1 文字
ax.text(
    arrow_pos_x + arrow_length / 2,
    arrow_pos_y - 0.04,
    "UMAP1",
    transform=ax.transAxes,
    ha="center",
    va="top",
    fontsize=12,
    fontfamily=arial_family
)

# UMAP2 文字
ax.text(
    arrow_pos_x - 0.04,
    arrow_pos_y + arrow_length / 2,
    "UMAP2",
    transform=ax.transAxes,
    ha="right",
    va="center",
    rotation=90,
    fontsize=12,
    fontfamily=arial_family
)

# ====================== 去掉坐标轴痕迹 ======================
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_xticks([])
ax.set_yticks([])
ax.set_title("")

for spine in ax.spines.values():
    spine.set_visible(False)

# ====================== 保存图片 ======================
fig.savefig(
    "Myeloid_UMAP_logcounts_Harmony_FigureC.pdf",
    bbox_inches="tight",
    transparent=True,
    dpi=300
)

plt.show()

In [29]:
adata_Myeloid.obs.head(1)

In [ ]:
adata_Myeloid.write_h5ad("/home/lishaogang/NEU2026_July/adata_final_Myeloid.h5ad")

In [27]:
dir()

In [28]:
adata.obs['cell_type'].unique()